In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json

# ============================================================
# STEP 3 — Formal split v1
# Train / Calibration / Validation / Frozen Test
# ============================================================

ROOT = Path(r"E:\GeoSIMCP_Reproduce\geoconformal-main")

DATA_PATH = (
    ROOT
    / "GCCP_PM2.5"
    / "00_data_audit"
    / "china_sites_2018_05_PM2.5_all.csv"
)

OUTPUT_DIR = ROOT / "GCCP_PM2.5" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

print("=" * 80)
print("STEP 3 — PM2.5 FORMAL SPLIT v1")
print("=" * 80)

# ============================================================
# 1. 读取数据
# ============================================================

data = pd.read_csv(
    DATA_PATH,
    encoding="gbk"
)

data = data.rename(
    columns={
        "监测点编码": "id",
        "监测点名称": "name",
        "城市": "city",
        "经度": "lon",
        "纬度": "lat",
    }
)

assert len(data) == 1415
assert data["id"].nunique() == 1415

FEATURES = [
    "dem",
    "sp",
    "tp",
    "blh",
    "r",
    "aod_sat",
    "ndvi",
]

TARGET = "PM2.5"

print("Data shape:", data.shape)

# ============================================================
# 2. 一次随机排列
#
# 不采用连续多次调参式 split，
# 而是一次性生成固定 permutation。
# ============================================================

rng = np.random.default_rng(SEED)

all_idx = np.arange(len(data))
perm = rng.permutation(all_idx)

# ------------------------------------------------------------
# 精确固定数量
# ------------------------------------------------------------

N_TRAIN = 990
N_CALIB = 141
N_VAL   = 142
N_TEST  = 142

assert (
    N_TRAIN
    + N_CALIB
    + N_VAL
    + N_TEST
    == len(data)
)

idx_train = perm[
    :N_TRAIN
]

idx_calib = perm[
    N_TRAIN:
    N_TRAIN + N_CALIB
]

idx_val = perm[
    N_TRAIN + N_CALIB:
    N_TRAIN + N_CALIB + N_VAL
]

idx_test = perm[
    N_TRAIN + N_CALIB + N_VAL:
]

# ============================================================
# 3. overlap 检查
# ============================================================

sets = {
    "train": set(idx_train),
    "calibration": set(idx_calib),
    "validation": set(idx_val),
    "test": set(idx_test),
}

names = list(sets.keys())

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        overlap = (
            sets[names[i]]
            & sets[names[j]]
        )

        assert len(overlap) == 0, (
            f"{names[i]} 与 {names[j]} "
            f"存在重叠"
        )

print("\n✅ 四个数据集无任何样本重叠")

# ============================================================
# 4. 创建 split assignment
# ============================================================

split_label = np.empty(
    len(data),
    dtype=object
)

split_label[idx_train] = "train"
split_label[idx_calib] = "calibration"
split_label[idx_val] = "validation"
split_label[idx_test] = "test"

split_df = pd.DataFrame({
    "original_index": np.arange(len(data)),
    "id": data["id"].astype(str),
    "name": data["name"],
    "city": data["city"],
    "split": split_label,
})

# 注意：
# 这里故意不把 PM2.5 写进 split 文件，
# 避免以后无意中查看 Frozen Test 标签。

# ============================================================
# 5. 打印数量
# ============================================================

print("\n" + "=" * 80)
print("FORMAL SPLIT COUNTS")
print("=" * 80)

counts = (
    split_df["split"]
    .value_counts()
)

for name in [
    "train",
    "calibration",
    "validation",
    "test",
]:
    n = counts[name]

    print(
        f"{name:<12s}: "
        f"{n:4d} "
        f"({n / len(data):.4%})"
    )

# ============================================================
# 6. 只打印站点 ID
#    不查看 Frozen Test 的 y
# ============================================================

print("\nTrain 前5个 ID:")
print(
    data.iloc[idx_train]["id"]
    .head()
    .tolist()
)

print("\nCalibration 前5个 ID:")
print(
    data.iloc[idx_calib]["id"]
    .head()
    .tolist()
)

print("\nValidation 前5个 ID:")
print(
    data.iloc[idx_val]["id"]
    .head()
    .tolist()
)

print("\nFrozen Test 前5个 ID:")
print(
    data.iloc[idx_test]["id"]
    .head()
    .tolist()
)

# ============================================================
# 7. 保存冻结 split
# ============================================================

SPLIT_PATH = (
    OUTPUT_DIR
    / "pm25_formal_split_assignment_v1.csv"
)

split_df.to_csv(
    SPLIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 8. 给 split 生成 SHA256 指纹
#
# 以后可以证明 split 没被偷偷换过。
# ============================================================

hash_text = "\n".join(
    f"{row.original_index}|{row.id}|{row.split}"
    for row in split_df.itertuples()
)

split_hash = hashlib.sha256(
    hash_text.encode("utf-8")
).hexdigest()

HASH_PATH = (
    OUTPUT_DIR
    / "pm25_formal_split_v1_sha256.txt"
)

HASH_PATH.write_text(
    split_hash,
    encoding="utf-8"
)

# ============================================================
# 9. 保存实验协议
# ============================================================

protocol = {
    "protocol_version": "pm25_formal_v1",
    "random_seed": SEED,

    "n_total": 1415,
    "n_train": N_TRAIN,
    "n_calibration": N_CALIB,
    "n_validation": N_VAL,
    "n_frozen_test": N_TEST,

    "features": FEATURES,

    "target": TARGET,

    # 来自已经完整复现成功的作者 RF
    # 从现在起固定，不根据正式 Test 调整
    "predictor": {
        "name": "RandomForestRegressor",
        "n_estimators": 300,
        "max_depth": None,
        "random_state": 42,
    },

    "alpha": 0.1,

    "test_policy": (
        "Frozen test labels must not be used "
        "for parameter selection."
    ),

    "split_sha256": split_hash,
}

PROTOCOL_PATH = (
    OUTPUT_DIR
    / "pm25_formal_protocol_v1.json"
)

with open(
    PROTOCOL_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        protocol,
        f,
        ensure_ascii=False,
        indent=2
    )

# ============================================================
# 10. 输出
# ============================================================

print("\n" + "=" * 80)
print("FROZEN SPLIT")
print("=" * 80)

print("Split CSV:")
print(SPLIT_PATH)

print("\nProtocol:")
print(PROTOCOL_PATH)

print("\nSHA256:")
print(split_hash)

print("\n✅ PM2.5 formal split v1 已冻结")
print("⚠️ 后续不得因为结果好坏重新划分 Test")

STEP 3 — PM2.5 FORMAL SPLIT v1
Data shape: (1415, 24)

✅ 四个数据集无任何样本重叠

FORMAL SPLIT COUNTS
train       :  990 (69.9647%)
calibration :  141 (9.9647%)
validation  :  142 (10.0353%)
test        :  142 (10.0353%)

Train 前5个 ID:
['1149A', '2268A', '1100A', '3065A', '1300A']

Calibration 前5个 ID:
['1004A', '1283A', '2558A', '1909A', '2272A']

Validation 前5个 ID:
['2429A', '2293A', '2551A', '1462A', '2536A']

Frozen Test 前5个 ID:
['1391A', '1109A', '2212A', '2384A', '2187A']

FROZEN SPLIT
Split CSV:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_split_assignment_v1.csv

Protocol:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_protocol_v1.json

SHA256:
785436f18bcc08a5a6b94916d3e544af0e0706e959b7c930300f7a91bddf04f3

✅ PM2.5 formal split v1 已冻结
⚠️ 后续不得因为结果好坏重新划分 Test


In [2]:
# ============================================================
# STEP 4 — Spatial coordinate audit
# China PM2.5
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import pairwise_distances
from sklearn.metrics.pairwise import haversine_distances

print("=" * 80)
print("STEP 4 — PM2.5 SPATIAL COORDINATE AUDIT")
print("=" * 80)

# ============================================================
# 1. 检查坐标字段
# ============================================================

coord_cols = [
    "lon",
    "lat",
    "proj_x",
    "proj_y"
]

print("\n坐标字段检查:")

for c in coord_cols:
    if c in data.columns:
        print(f"✅ {c}")
    else:
        print(f"❌ {c}")

if not all(c in data.columns for c in coord_cols):
    raise RuntimeError(
        "lon/lat/proj_x/proj_y 不完整"
    )

print("\n缺失值:")
print(
    data[coord_cols]
    .isna()
    .sum()
)

print("\n坐标统计:")
display(
    data[coord_cols]
    .describe()
    .T
)

# ============================================================
# 2. 检查 projected coordinates 数值范围
# ============================================================

print("\n" + "=" * 80)
print("PROJECTED COORDINATE RANGE")
print("=" * 80)

for c in ["proj_x", "proj_y"]:
    print(
        f"{c}: "
        f"min={data[c].min():.3f}, "
        f"max={data[c].max():.3f}, "
        f"range={data[c].max()-data[c].min():.3f}"
    )

# ============================================================
# 3. 计算所有站点间 Haversine 距离
#
# sklearn haversine:
# 输入必须是 [lat, lon] radians
# 输出是球心夹角 radians
# 乘地球半径得到 km
# ============================================================

EARTH_RADIUS_KM = 6371.0088

latlon_rad = np.radians(
    data[["lat", "lon"]].to_numpy(dtype=float)
)

D_hav_km = (
    haversine_distances(latlon_rad)
    * EARTH_RADIUS_KM
)

# ============================================================
# 4. projected Euclidean distance
# ============================================================

proj = data[
    ["proj_x", "proj_y"]
].to_numpy(dtype=float)

D_proj = pairwise_distances(
    proj,
    metric="euclidean"
)

# ============================================================
# 5. 只取上三角，避免重复和对角线
# ============================================================

iu = np.triu_indices(
    len(data),
    k=1
)

hav_km = D_hav_km[iu]
proj_dist = D_proj[iu]

print("\nPair 数量:", len(hav_km))

# ============================================================
# 6. 判断 proj_x/proj_y 的尺度
#
# 分别假设：
# proj 单位是 meter
# 或 km
# ============================================================

proj_km_if_meter = (
    proj_dist / 1000.0
)

# ============================================================
# 7. 距离分布
# ============================================================

def qsummary(x):
    return {
        "min": np.min(x),
        "p01": np.quantile(x, 0.01),
        "p05": np.quantile(x, 0.05),
        "p25": np.quantile(x, 0.25),
        "median": np.median(x),
        "p75": np.quantile(x, 0.75),
        "p95": np.quantile(x, 0.95),
        "p99": np.quantile(x, 0.99),
        "max": np.max(x),
    }

distance_summary = pd.DataFrame({
    "Haversine_km": qsummary(hav_km),
    "Projected_if_meter_km": qsummary(
        proj_km_if_meter
    ),
}).T

print("\n" + "=" * 80)
print("PAIRWISE DISTANCE SUMMARY")
print("=" * 80)

display(distance_summary)

# ============================================================
# 8. Haversine vs projected correlation
# ============================================================

pearson_corr = np.corrcoef(
    hav_km,
    proj_km_if_meter
)[0, 1]

spearman_corr = pd.Series(
    hav_km
).corr(
    pd.Series(proj_km_if_meter),
    method="spearman"
)

ratio = (
    proj_km_if_meter
    / np.maximum(hav_km, 1e-12)
)

print("\n" + "=" * 80)
print("HAVERSINE vs PROJECTED")
print("=" * 80)

print(
    "Pearson correlation :",
    pearson_corr
)

print(
    "Spearman correlation:",
    spearman_corr
)

print("\nProjected/Haversine distance ratio:")

print(
    "median:",
    np.median(ratio)
)

print(
    "p05   :",
    np.quantile(ratio, 0.05)
)

print(
    "p95   :",
    np.quantile(ratio, 0.95)
)

# ============================================================
# 9. 特别检查近距离站点
#
# 全国尺度相关性高不代表局部也准确。
# GeoCP/GCCP 最关注近邻。
# ============================================================

near_masks = {
    "<50 km": hav_km < 50,
    "<100 km": hav_km < 100,
    "<200 km": hav_km < 200,
    "<500 km": hav_km < 500,
}

print("\n" + "=" * 80)
print("LOCAL DISTANCE AGREEMENT")
print("=" * 80)

local_rows = []

for label, mask in near_masks.items():

    n = np.sum(mask)

    if n == 0:
        continue

    r = (
        proj_km_if_meter[mask]
        / np.maximum(
            hav_km[mask],
            1e-12
        )
    )

    corr = np.corrcoef(
        hav_km[mask],
        proj_km_if_meter[mask]
    )[0, 1]

    local_rows.append({
        "distance_group": label,
        "n_pairs": n,
        "pearson": corr,
        "median_ratio": np.median(r),
        "p05_ratio": np.quantile(r, 0.05),
        "p95_ratio": np.quantile(r, 0.95),
    })

local_df = pd.DataFrame(local_rows)

display(local_df)

# ============================================================
# 10. 最近邻尺度
# ============================================================

D_hav_nn = D_hav_km.copy()
np.fill_diagonal(
    D_hav_nn,
    np.inf
)

nearest_hav = np.min(
    D_hav_nn,
    axis=1
)

D_proj_nn = (
    D_proj.copy()
    / 1000.0
)

np.fill_diagonal(
    D_proj_nn,
    np.inf
)

nearest_proj = np.min(
    D_proj_nn,
    axis=1
)

nn_df = pd.DataFrame({
    "nearest_haversine_km":
        nearest_hav,

    "nearest_projected_km":
        nearest_proj,
})

print("\n" + "=" * 80)
print("NEAREST-NEIGHBOR DISTANCE")
print("=" * 80)

display(
    nn_df.describe(
        percentiles=[
            0.05,
            0.25,
            0.50,
            0.75,
            0.95
        ]
    ).T
)

print(
    "\n最近邻距离相关性:",
    np.corrcoef(
        nearest_hav,
        nearest_proj
    )[0, 1]
)

# ============================================================
# 11. 保存审计结果
# ============================================================

distance_summary.to_csv(
    OUTPUT_DIR
    / "pm25_spatial_distance_summary_v1.csv",
    encoding="utf-8-sig",
)

local_df.to_csv(
    OUTPUT_DIR
    / "pm25_spatial_local_agreement_v1.csv",
    index=False,
    encoding="utf-8-sig",
)

nn_df.to_csv(
    OUTPUT_DIR
    / "pm25_spatial_nearest_neighbor_v1.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n✅ STEP 4 完成")

STEP 4 — PM2.5 SPATIAL COORDINATE AUDIT

坐标字段检查:
✅ lon
✅ lat
✅ proj_x
✅ proj_y

缺失值:
lon       0
lat       0
proj_x    0
proj_y    0
dtype: int64

坐标统计:


,count,mean,std,min,25%,50%,75%,max
lon,1415.0,1.133238e+02,8.444840,7.594350e+01,1.090628e+02,1.142860e+02,1.188290e+02,1.311651e+02
lat,1415.0,3.318643e+01,6.357294,1.995070e+01,2.859330e+01,3.218889e+01,3.782225e+01,5.042690e+01
proj_x,1415.0,1.548435e+06,656555.168304,-1.484910e+06,1.186948e+06,1.684335e+06,2.033587e+06,2.504442e+06
proj_y,1415.0,5.602481e+05,801468.700610,-1.044494e+06,-3.173230e+04,4.930251e+05,1.089367e+06,2.759831e+06



PROJECTED COORDINATE RANGE
proj_x: min=-1484909.609, max=2504442.429, range=3989352.038
proj_y: min=-1044494.260, max=2759831.032, range=3804325.293

Pair 数量: 1000405

PAIRWISE DISTANCE SUMMARY


,min,p01,p05,p25,median,p75,p95,p99,max
Haversine_km,0.255749,96.942142,267.517451,704.868901,1134.396699,1644.774518,2793.775460,3492.279591,4561.084720
Projected_if_meter_km,0.278733,96.681828,267.684863,707.023226,1144.958667,1666.480546,2753.051726,3387.295933,4284.845152



HAVERSINE vs PROJECTED
Pearson correlation : 0.9952494072326912
Spearman correlation: 0.9963808514258791

Projected/Haversine distance ratio:
median: 1.0088180703593073
p05   : 0.927237115241472
p95   : 1.0791882097083048

LOCAL DISTANCE AGREEMENT


,distance_group,n_pairs,pearson,median_ratio,p05_ratio,p95_ratio
0,<50 km,4403,0.996685,1.000128,0.922822,1.083305
1,<100 km,10444,0.994751,1.001264,0.923914,1.083427
2,<200 km,31429,0.992009,0.998853,0.923527,1.081728
3,<500 km,139838,0.990776,1.004349,0.923918,1.083014



NEAREST-NEIGHBOR DISTANCE


,count,mean,std,min,5%,25%,50%,75%,95%,max
nearest_haversine_km,1415.0,7.124904,17.430320,0.255749,1.461110,2.394444,3.676150,5.903105,20.571573,278.085470
nearest_projected_km,1415.0,7.158415,17.845206,0.278733,1.426847,2.415125,3.649117,5.912607,20.391727,291.134846



最近邻距离相关性: 0.998836087480368

✅ STEP 4 完成


In [3]:
# ============================================================
# STEP 5 — Formal predictor
# Random Forest
#
# 重要：
# Frozen Test 不参与任何模型选择或指标计算
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

print("=" * 80)
print("STEP 5 — PM2.5 FORMAL RANDOM FOREST")
print("=" * 80)

FEATURES = [
    "dem",
    "sp",
    "tp",
    "blh",
    "r",
    "aod_sat",
    "ndvi",
]

MODEL_FEATURES = FEATURES + [
    "lon",
    "lat",
]

# ============================================================
# 1. 根据已经冻结的 index 取数据
# ============================================================

X_train = (
    data.iloc[idx_train]
    [MODEL_FEATURES]
    .copy()
)

X_calib = (
    data.iloc[idx_calib]
    [MODEL_FEATURES]
    .copy()
)

X_val = (
    data.iloc[idx_val]
    [MODEL_FEATURES]
    .copy()
)

# Test X 可以准备，
# 但现在不进行任何 Test 性能评估
X_test = (
    data.iloc[idx_test]
    [MODEL_FEATURES]
    .copy()
)

y_train = (
    data.iloc[idx_train]
    ["PM2.5"]
    .copy()
)

y_calib = (
    data.iloc[idx_calib]
    ["PM2.5"]
    .copy()
)

y_val = (
    data.iloc[idx_val]
    ["PM2.5"]
    .copy()
)

# ------------------------------------------------------------
# 注意：
# 此处故意不定义 y_test
# 防止后面误用 Frozen Test 标签
# ------------------------------------------------------------

# 非空间特征，后面 GeoSIMCP / GCCP 使用
X_aspatial_train = (
    data.iloc[idx_train]
    [FEATURES]
    .copy()
)

X_aspatial_calib = (
    data.iloc[idx_calib]
    [FEATURES]
    .copy()
)

X_aspatial_val = (
    data.iloc[idx_val]
    [FEATURES]
    .copy()
)

X_aspatial_test = (
    data.iloc[idx_test]
    [FEATURES]
    .copy()
)

# 坐标
coord_train_lonlat = (
    data.iloc[idx_train]
    [["lon", "lat"]]
    .copy()
)

coord_calib_lonlat = (
    data.iloc[idx_calib]
    [["lon", "lat"]]
    .copy()
)

coord_val_lonlat = (
    data.iloc[idx_val]
    [["lon", "lat"]]
    .copy()
)

coord_test_lonlat = (
    data.iloc[idx_test]
    [["lon", "lat"]]
    .copy()
)

coord_train_proj = (
    data.iloc[idx_train]
    [["proj_x", "proj_y"]]
    .copy()
)

# ============================================================
# 2. 固定 Random Forest
# ============================================================

formal_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)

formal_rf.fit(
    X_train,
    y_train
)

# ============================================================
# 3. 只评估 Train / Calibration / Validation
# ============================================================

def metrics(y_true, y_pred):
    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return rmse, mae, r2


pred_train = formal_rf.predict(
    X_train
)

pred_calib = formal_rf.predict(
    X_calib
)

pred_val = formal_rf.predict(
    X_val
)

m_train = metrics(
    y_train,
    pred_train
)

m_calib = metrics(
    y_calib,
    pred_calib
)

m_val = metrics(
    y_val,
    pred_val
)

metric_df = pd.DataFrame(
    [
        [
            "Train",
            len(y_train),
            *m_train
        ],
        [
            "Calibration",
            len(y_calib),
            *m_calib
        ],
        [
            "Validation",
            len(y_val),
            *m_val
        ],
    ],
    columns=[
        "split",
        "n",
        "RMSE",
        "MAE",
        "R2",
    ]
)

print("\nPrediction performance:")
display(metric_df)

# ============================================================
# 4. Calibration residual
#
# residual（残差）：
# r_i = |y_i - yhat_i|
#
# 后面所有 CP 方法都使用同一组 residual
# ============================================================

calib_residual = np.abs(
    y_calib.to_numpy()
    - pred_calib
)

print("\n" + "=" * 80)
print("CALIBRATION RESIDUAL")
print("=" * 80)

print(
    pd.Series(
        calib_residual
    ).describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

# ============================================================
# 5. Validation prediction 固定保存
# ============================================================

val_pred_df = pd.DataFrame({
    "original_index":
        idx_val,

    "id":
        data.iloc[idx_val]["id"].to_numpy(),

    "y_true":
        y_val.to_numpy(),

    "y_pred":
        pred_val,
})

val_pred_df[
    "abs_error"
] = np.abs(
    val_pred_df["y_true"]
    - val_pred_df["y_pred"]
)

VAL_PATH = (
    OUTPUT_DIR
    / "pm25_formal_validation_predictions_v1.csv"
)

val_pred_df.to_csv(
    VAL_PATH,
    index=False,
    encoding="utf-8-sig"
)

METRIC_PATH = (
    OUTPUT_DIR
    / "pm25_formal_rf_non_test_metrics_v1.csv"
)

metric_df.to_csv(
    METRIC_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n保存:")
print(VAL_PATH)
print(METRIC_PATH)

print("\n✅ STEP 5 完成")
print(
    "⚠️ Frozen Test y 尚未读取、"
    "尚未计算任何 Test 指标"
)

STEP 5 — PM2.5 FORMAL RANDOM FOREST

Prediction performance:


,split,n,RMSE,MAE,R2
0,Train,990,2.818152,1.711510,0.953701
1,Calibration,141,5.201787,3.964660,0.772227
2,Validation,142,7.990551,5.344817,0.597652



CALIBRATION RESIDUAL
count    141.000000
mean       3.964660
std        3.379506
min        0.004517
50%        3.167240
75%        5.827628
90%        8.953932
95%       10.249484
99%       13.997740
max       14.813640
dtype: float64

保存:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_validation_predictions_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_rf_non_test_metrics_v1.csv

✅ STEP 5 完成
⚠️ Frozen Test y 尚未读取、尚未计算任何 Test 指标


In [4]:
# ============================================================
# STEP 5B — Formal RF selection using TRAIN ONLY
#
# 重要：
# 1. 超参数只在 Formal Train=990 内部 3-fold CV
# 2. Calibration / Validation 不参与 RF 参数选择
# 3. Frozen Test y 完全不读取
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

print("=" * 80)
print("STEP 5B — FORMAL RF: TRAIN-ONLY HYPERPARAMETER SELECTION")
print("=" * 80)

# ============================================================
# 1. 与作者完全相同的 RF 搜索空间
# ============================================================

rf_base = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [None, 10],
}

rf_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=-1,
    return_train_score=True
)

# 只使用 Formal Train
rf_search.fit(
    X_train,
    y_train
)

print("\nBest parameters:")
print(rf_search.best_params_)

print(
    "Best CV RMSE:",
    np.sqrt(-rf_search.best_score_)
)

# ============================================================
# 2. 正式 predictor
# ============================================================

formal_rf = rf_search.best_estimator_

# ============================================================
# 3. 只评价 Train / Calibration / Validation
# ============================================================

pred_train = formal_rf.predict(X_train)
pred_calib = formal_rf.predict(X_calib)
pred_val   = formal_rf.predict(X_val)


def regression_metrics(y_true, y_pred):
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    mae = mean_absolute_error(
        y_true, y_pred
    )
    r2 = r2_score(
        y_true, y_pred
    )
    return rmse, mae, r2


rows = []

for split_name, yy, pp in [
    ("Train", y_train, pred_train),
    ("Calibration", y_calib, pred_calib),
    ("Validation", y_val, pred_val),
]:
    rmse, mae, r2 = regression_metrics(yy, pp)

    rows.append({
        "split": split_name,
        "n": len(yy),
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

formal_metric_df = pd.DataFrame(rows)

print("\nFormal predictor performance:")
display(formal_metric_df)

# ============================================================
# 4. 检查非 Test target 分布
#
# 只检查 Train / Calibration / Validation
# 不读取 Frozen Test y
# ============================================================

target_audit = pd.DataFrame({
    "Train": y_train.describe(),
    "Calibration": y_calib.describe(),
    "Validation": y_val.describe(),
}).T

print("\n" + "=" * 80)
print("NON-TEST PM2.5 DISTRIBUTION")
print("=" * 80)

display(target_audit)

# ============================================================
# 5. 正式 Calibration residual
# ============================================================

calib_residual = np.abs(
    y_calib.to_numpy(dtype=float)
    - pred_calib
)

print("\n" + "=" * 80)
print("FORMAL CALIBRATION RESIDUAL")
print("=" * 80)

display(
    pd.Series(
        calib_residual,
        name="abs_residual"
    ).describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

# ============================================================
# 6. 保存正式 RF 结果
# ============================================================

rf_summary = pd.DataFrame([{
    "protocol": "pm25_formal_v1",
    "selection_data": "train_only",
    "cv": 3,
    "scoring": "neg_mean_squared_error",
    "best_n_estimators":
        rf_search.best_params_["n_estimators"],
    "best_max_depth":
        str(rf_search.best_params_["max_depth"]),
    "best_cv_rmse":
        np.sqrt(-rf_search.best_score_),
}])

rf_summary.to_csv(
    OUTPUT_DIR
    / "pm25_formal_rf_train_only_selection_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

formal_metric_df.to_csv(
    OUTPUT_DIR
    / "pm25_formal_rf_non_test_metrics_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 5B 完成")
print("⚠️ Frozen Test y 仍未读取")

STEP 5B — FORMAL RF: TRAIN-ONLY HYPERPARAMETER SELECTION

Best parameters:
{'max_depth': None, 'n_estimators': 300}
Best CV RMSE: 7.825127510685215

Formal predictor performance:


,split,n,RMSE,MAE,R2
0,Train,990,2.818152,1.711510,0.953701
1,Calibration,141,5.201787,3.964660,0.772227
2,Validation,142,7.990551,5.344817,0.597652



NON-TEST PM2.5 DISTRIBUTION


,count,mean,std,min,25%,50%,75%,max
Train,990.0,33.172300,13.103830,3.112903,25.584677,32.590054,40.050000,191.620690
Calibration,141.0,32.699759,10.938234,6.677419,25.596774,31.666667,40.274194,59.066667
Validation,142.0,33.433313,12.641847,3.983333,25.650000,32.261350,41.489144,85.683333



FORMAL CALIBRATION RESIDUAL


count    141.000000
mean       3.964660
std        3.379506
min        0.004517
50%        3.167240
75%        5.827628
90%        8.953932
95%       10.249484
99%       13.997740
max       14.813640
Name: abs_residual, dtype: float64


✅ STEP 5B 完成
⚠️ Frozen Test y 仍未读取


In [5]:
# ============================================================
# STEP 6 — Global Split Conformal Prediction
# Validation only
# Frozen Test 不使用
# ============================================================

import math
import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 6 — GLOBAL SPLIT CONFORMAL PREDICTION")
print("=" * 80)

ALPHA = 0.1

# ------------------------------------------------------------
# 1. Calibration residual
# r_i = |y_i - yhat_i|
# ------------------------------------------------------------

scores = np.asarray(
    calib_residual,
    dtype=float
)

n_calib = len(scores)

# ------------------------------------------------------------
# 2. 有限样本 conformal quantile
#
# k = ceil((n+1)*(1-alpha))
# ------------------------------------------------------------

k = math.ceil(
    (n_calib + 1) * (1 - ALPHA)
)

k = min(k, n_calib)

scores_sorted = np.sort(scores)

q_cp = scores_sorted[k - 1]

print("Calibration n =", n_calib)
print("alpha         =", ALPHA)
print("k             =", k)
print("q_CP          =", q_cp)

# ------------------------------------------------------------
# 3. Validation prediction interval
# ------------------------------------------------------------

val_pred = np.asarray(
    pred_val,
    dtype=float
)

val_true = np.asarray(
    y_val,
    dtype=float
)

lower = val_pred - q_cp
upper = val_pred + q_cp

# ------------------------------------------------------------
# 4. Coverage
# ------------------------------------------------------------

covered = (
    (val_true >= lower)
    & (val_true <= upper)
)

coverage = np.mean(covered)

# ------------------------------------------------------------
# 5. Width
# ------------------------------------------------------------

width = upper - lower

mean_half_width = q_cp
mean_width = np.mean(width)

# ------------------------------------------------------------
# 6. Interval Score
# ------------------------------------------------------------

def interval_score(
    y_true,
    lower,
    upper,
    alpha=0.1
):
    y_true = np.asarray(y_true, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)

    width = upper - lower

    below = (
        (lower - y_true)
        * (y_true < lower)
    )

    above = (
        (y_true - upper)
        * (y_true > upper)
    )

    return (
        width
        + (2 / alpha) * below
        + (2 / alpha) * above
    )


val_interval_score = interval_score(
    val_true,
    lower,
    upper,
    alpha=ALPHA
)

mean_interval_score = np.mean(
    val_interval_score
)

# ------------------------------------------------------------
# 7. 输出
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("GLOBAL CP — VALIDATION")
print("=" * 80)

print(f"Coverage        = {coverage:.6f}")
print(f"Mean Half-width = {mean_half_width:.6f}")
print(f"Mean Width      = {mean_width:.6f}")
print(f"Interval Score  = {mean_interval_score:.6f}")
print(f"Covered         = {covered.sum()} / {len(covered)}")

# ------------------------------------------------------------
# 8. 保存
# ------------------------------------------------------------

cp_val_df = pd.DataFrame({
    "original_index": idx_val,
    "id": data.iloc[idx_val]["id"].to_numpy(),
    "y_true": val_true,
    "y_pred": val_pred,
    "lower": lower,
    "upper": upper,
    "width": width,
    "covered": covered.astype(int),
    "interval_score": val_interval_score,
})

cp_val_df.to_csv(
    OUTPUT_DIR
    / "pm25_formal_cp_validation_per_sample_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

cp_summary = pd.DataFrame([{
    "method": "CP",
    "alpha": ALPHA,
    "n_calibration": n_calib,
    "conformal_k": k,
    "q": q_cp,
    "coverage": coverage,
    "mean_half_width": mean_half_width,
    "mean_width": mean_width,
    "interval_score": mean_interval_score,
}])

cp_summary.to_csv(
    OUTPUT_DIR
    / "pm25_formal_cp_validation_summary_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 6 完成")
print("⚠️ Frozen Test 仍未读取")

STEP 6 — GLOBAL SPLIT CONFORMAL PREDICTION
Calibration n = 141
alpha         = 0.1
k             = 128
q_CP          = 9.133607116712255

GLOBAL CP — VALIDATION
Coverage        = 0.859155
Mean Half-width = 9.133607
Mean Width      = 18.267214
Interval Score  = 34.808273
Covered         = 122 / 142

✅ STEP 6 完成
⚠️ Frozen Test 仍未读取


In [6]:
# ============================================================
# STEP 7 — Formal GeoCP
# Haversine geographic distance
# Parameter selection: Validation ONLY
# Frozen Test y is NOT used
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import haversine_distances

print("=" * 80)
print("STEP 7 — FORMAL GeoCP / VALIDATION BANDWIDTH SEARCH")
print("=" * 80)

ALPHA = 0.1
TARGET_COVERAGE = 0.90
EARTH_RADIUS_KM = 6371.0088

# ============================================================
# 1. Calibration / Validation geographic coordinates
#
# sklearn haversine requires [lat, lon] in radians
# ============================================================

coord_calib_latlon = (
    data.iloc[idx_calib]
    [["lat", "lon"]]
    .to_numpy(dtype=float)
)

coord_val_latlon = (
    data.iloc[idx_val]
    [["lat", "lon"]]
    .to_numpy(dtype=float)
)

coord_calib_rad = np.radians(
    coord_calib_latlon
)

coord_val_rad = np.radians(
    coord_val_latlon
)

# Validation x Calibration distance matrix
D_geo_val_calib = (
    haversine_distances(
        coord_val_rad,
        coord_calib_rad
    )
    * EARTH_RADIUS_KM
)

print(
    "Distance matrix shape:",
    D_geo_val_calib.shape
)

assert D_geo_val_calib.shape == (
    len(idx_val),
    len(idx_calib)
)

# ============================================================
# 2. Geographic distance scale audit
#
# 只使用 coordinates，不使用 y
# ============================================================

nearest_calib_km = (
    D_geo_val_calib.min(axis=1)
)

all_geo_dist = (
    D_geo_val_calib.ravel()
)

print("\n" + "=" * 80)
print("VALIDATION → CALIBRATION DISTANCE SCALE")
print("=" * 80)

print(
    "Nearest calibration distance (km):"
)

print(
    pd.Series(
        nearest_calib_km
    ).describe(
        percentiles=[
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)

print(
    "\nAll validation-calibration distances (km):"
)

print(
    pd.Series(
        all_geo_dist
    ).describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)

# ============================================================
# 3. Freeze geographic bandwidth grid
#
# Rule:
# lower = 0.5 × 5th percentile of nearest-calibration distance
# upper = 90th percentile of all Val-Calib distances
#
# 完全由坐标确定，不看 Validation y
# ============================================================

h_min = max(
    1.0,
    0.5 * np.quantile(
        nearest_calib_km,
        0.05
    )
)

h_max = np.quantile(
    all_geo_dist,
    0.90
)

H_GEO_GRID = np.geomspace(
    h_min,
    h_max,
    25
)

print("\n" + "=" * 80)
print("FROZEN GEO BANDWIDTH GRID")
print("=" * 80)

print("h_min =", h_min, "km")
print("h_max =", h_max, "km")
print("n_grid =", len(H_GEO_GRID))

print("\nBandwidths:")
print(
    np.round(
        H_GEO_GRID,
        3
    )
)

# ============================================================
# 4. Stable Gaussian kernel
#
# log-weight shifting prevents numerical underflow:
#
# w_i ∝ exp[-0.5 (d_i/h)^2]
#
# Multiplying all weights by the same constant does NOT
# change the weighted quantile.
# ============================================================

def stable_gaussian_weights(
    distances,
    bandwidth
):
    d = np.asarray(
        distances,
        dtype=float
    )

    log_w = (
        -0.5
        * (d / bandwidth) ** 2
    )

    # 数值稳定处理
    log_w = (
        log_w
        - np.max(log_w)
    )

    w = np.exp(log_w)

    return w


# ============================================================
# 5. Weighted quantile
#
# 与 GeoCP / GeoSIMCP 作者逻辑保持一致：
#
# residual 从小到大排序，
# 累积 similarity weights，
# 第一次达到 1-alpha 时的 residual 即 q*
# ============================================================

def weighted_quantile_local(
    residuals,
    weights,
    alpha=0.1
):
    residuals = np.asarray(
        residuals,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    order = np.argsort(
        residuals
    )

    s = residuals[order]
    w = weights[order]

    total = w.sum()

    if (
        not np.isfinite(total)
        or total <= 0
    ):
        raise RuntimeError(
            "Invalid kernel weights"
        )

    cumulative = (
        np.cumsum(w)
        / total
    )

    idx = np.searchsorted(
        cumulative,
        1 - alpha,
        side="left"
    )

    idx = min(
        idx,
        len(s) - 1
    )

    return s[idx]


# ============================================================
# 6. Effective sample size
#
# n_eff = (sum w)^2 / sum(w^2)
#
# 越大代表使用的 Calibration 样本越“全局”
# 越小代表越局部。
# ============================================================

def effective_sample_size(w):

    w = np.asarray(
        w,
        dtype=float
    )

    return (
        w.sum() ** 2
        / np.sum(w ** 2)
    )


# ============================================================
# 7. Interval Score
# ============================================================

def interval_score(
    y_true,
    lower,
    upper,
    alpha=0.1
):
    y_true = np.asarray(
        y_true,
        dtype=float
    )

    lower = np.asarray(
        lower,
        dtype=float
    )

    upper = np.asarray(
        upper,
        dtype=float
    )

    width = upper - lower

    below = (
        (lower - y_true)
        * (y_true < lower)
    )

    above = (
        (y_true - upper)
        * (y_true > upper)
    )

    return (
        width
        + (2 / alpha) * below
        + (2 / alpha) * above
    )


# ============================================================
# 8. Calibration residuals + Validation predictions
# ============================================================

residuals = np.asarray(
    calib_residual,
    dtype=float
)

val_pred = np.asarray(
    pred_val,
    dtype=float
)

val_true = np.asarray(
    y_val,
    dtype=float
)

# ============================================================
# 9. Bandwidth sweep
# ============================================================

rows = []
results_by_h = {}

for h in H_GEO_GRID:

    q_local = np.zeros(
        len(idx_val),
        dtype=float
    )

    neff = np.zeros(
        len(idx_val),
        dtype=float
    )

    for j in range(
        len(idx_val)
    ):

        w = stable_gaussian_weights(
            D_geo_val_calib[j],
            h
        )

        q_local[j] = (
            weighted_quantile_local(
                residuals,
                w,
                alpha=ALPHA
            )
        )

        neff[j] = (
            effective_sample_size(w)
        )

    lower = (
        val_pred
        - q_local
    )

    upper = (
        val_pred
        + q_local
    )

    covered = (
        (val_true >= lower)
        & (val_true <= upper)
    )

    width = (
        upper - lower
    )

    iscore = interval_score(
        val_true,
        lower,
        upper,
        alpha=ALPHA
    )

    row = {
        "bandwidth_km": h,
        "coverage": np.mean(
            covered
        ),
        "mean_half_width":
            np.mean(q_local),
        "mean_width":
            np.mean(width),
        "interval_score":
            np.mean(iscore),

        "median_n_eff":
            np.median(neff),

        "p10_n_eff":
            np.quantile(
                neff,
                0.10
            ),

        "p90_n_eff":
            np.quantile(
                neff,
                0.90
            ),

        "min_n_eff":
            np.min(neff),

        "max_n_eff":
            np.max(neff),
    }

    rows.append(row)

    results_by_h[h] = {
        "q_local": q_local,
        "lower": lower,
        "upper": upper,
        "covered": covered,
        "width": width,
        "interval_score": iscore,
        "n_eff": neff,
    }


geocp_grid_df = pd.DataFrame(
    rows
)

# ============================================================
# 10. Validation parameter selection
#
# Rule frozen beforehand:
#
# Coverage >= 0.90
#          ↓
# minimum Interval Score
#
# Test is NOT involved.
# ============================================================

feasible = geocp_grid_df[
    geocp_grid_df["coverage"]
    >= TARGET_COVERAGE
].copy()

print("\n" + "=" * 80)
print("GeoCP GRID RESULT")
print("=" * 80)

display(
    geocp_grid_df
)

if len(feasible) == 0:

    print(
        "\n❌ 当前 grid 中没有 "
        "Validation Coverage >= 0.90 的 bandwidth"
    )

    print(
        "\nMaximum Validation Coverage:"
    )

    display(
        geocp_grid_df.sort_values(
            "coverage",
            ascending=False
        ).head(5)
    )

    raise RuntimeError(
        "No feasible GeoCP bandwidth. "
        "Do NOT choose a sub-0.90 bandwidth. "
        "Review grid range first."
    )


best_idx = (
    feasible[
        "interval_score"
    ].idxmin()
)

best_geo = (
    geocp_grid_df
    .loc[best_idx]
)

best_h_geo = float(
    best_geo["bandwidth_km"]
)

best_geo_result = (
    results_by_h[
        best_h_geo
    ]
)

print("\n" + "=" * 80)
print("FORMAL GeoCP — BEST VALIDATION PARAMETER")
print("=" * 80)

print(
    f"Bandwidth (km)  = "
    f"{best_h_geo:.6f}"
)

print(
    f"Coverage        = "
    f"{best_geo['coverage']:.6f}"
)

print(
    f"Mean Half-width = "
    f"{best_geo['mean_half_width']:.6f}"
)

print(
    f"Mean Width      = "
    f"{best_geo['mean_width']:.6f}"
)

print(
    f"Interval Score  = "
    f"{best_geo['interval_score']:.6f}"
)

print(
    f"Median n_eff    = "
    f"{best_geo['median_n_eff']:.6f}"
)

print(
    f"p10 n_eff       = "
    f"{best_geo['p10_n_eff']:.6f}"
)

print(
    f"p90 n_eff       = "
    f"{best_geo['p90_n_eff']:.6f}"
)

# ============================================================
# 11. Compare with Global CP on VALIDATION only
# ============================================================

print("\n" + "=" * 80)
print("VALIDATION: CP vs GeoCP")
print("=" * 80)

comparison_val = pd.DataFrame([
    {
        "method": "CP",
        "coverage": coverage,
        "mean_width": mean_width,
        "interval_score":
            mean_interval_score,
        "median_n_eff":
            float(n_calib),
    },

    {
        "method": "GeoCP",
        "coverage":
            best_geo["coverage"],
        "mean_width":
            best_geo["mean_width"],
        "interval_score":
            best_geo["interval_score"],
        "median_n_eff":
            best_geo["median_n_eff"],
    },
])

display(
    comparison_val
)

# ============================================================
# 12. Save grid and frozen selected parameter
# ============================================================

geocp_grid_df.to_csv(
    OUTPUT_DIR
    / "pm25_formal_geocp_validation_grid_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([
    best_geo
]).to_csv(
    OUTPUT_DIR
    / "pm25_formal_geocp_validation_best_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "original_index": idx_val,
    "id":
        data.iloc[idx_val]["id"]
        .to_numpy(),

    "y_true":
        val_true,

    "y_pred":
        val_pred,

    "q_local":
        best_geo_result[
            "q_local"
        ],

    "lower":
        best_geo_result[
            "lower"
        ],

    "upper":
        best_geo_result[
            "upper"
        ],

    "width":
        best_geo_result[
            "width"
        ],

    "covered":
        best_geo_result[
            "covered"
        ].astype(int),

    "interval_score":
        best_geo_result[
            "interval_score"
        ],

    "n_eff":
        best_geo_result[
            "n_eff"
        ],
}).to_csv(
    OUTPUT_DIR
    / "pm25_formal_geocp_validation_per_sample_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

# 保存这个 grid，后面 Additive-EUC / GCCP-B
# 统一使用完全相同的 geographic bandwidth candidates
pd.DataFrame({
    "h_geo_km":
        H_GEO_GRID
}).to_csv(
    OUTPUT_DIR
    / "pm25_formal_h_geo_grid_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 7 完成")
print("✅ best_h_geo 已根据 Validation 冻结")
print("⚠️ Frozen Test y 仍未读取")

STEP 7 — FORMAL GeoCP / VALIDATION BANDWIDTH SEARCH
Distance matrix shape: (142, 141)

VALIDATION → CALIBRATION DISTANCE SCALE
Nearest calibration distance (km):
count     142.000000
mean      106.398282
std       130.358069
min         1.173562
5%          3.153724
10%         4.696160
25%        18.510412
50%        83.477739
75%       155.254753
90%       226.391093
95%       275.561312
max      1134.476470
dtype: float64

All validation-calibration distances (km):
count    20022.000000
mean      1278.491374
std        762.025283
min          1.173562
1%         100.357899
5%         274.744345
25%        715.058524
50%       1151.932686
75%       1673.694721
90%       2331.631595
95%       2817.921713
max       4543.550779
dtype: float64

FROZEN GEO BANDWIDTH GRID
h_min = 1.5768620734001186 km
h_max = 2331.6315954622364 km
n_grid = 25

Bandwidths:
[1.577000e+00 2.137000e+00 2.897000e+00 3.927000e+00 5.322000e+00
 7.214000e+00 9.778000e+00 1.325400e+01 1.796500e+01 2.435000e+01
 3.3

,bandwidth_km,coverage,mean_half_width,mean_width,interval_score,median_n_eff,p10_n_eff,p90_n_eff,min_n_eff,max_n_eff
0,1.576862,0.429577,4.169711,8.339421,61.876287,1.000000,1.000000,1.000000,1.000000,1.528262
1,2.137329,0.436620,4.192335,8.384670,61.529014,1.000000,1.000000,1.000000,1.000000,1.805367
2,2.897005,0.436620,4.192335,8.384670,61.529014,1.000000,1.000000,1.000000,1.000000,1.934856
3,3.926693,0.436620,4.192335,8.384670,61.529014,1.000000,1.000000,1.000000,1.000000,1.979932
4,5.322366,0.436620,4.232045,8.464090,61.316947,1.000000,1.000000,1.000002,1.000000,2.104075
5,7.214106,0.443662,4.300450,8.600900,61.417138,1.000000,1.000000,1.001054,1.000000,2.436618
6,9.778230,0.450704,4.335237,8.670475,61.034539,1.000000,1.000000,1.032831,1.000000,2.743033
7,13.253727,0.457746,4.340541,8.681082,61.014237,1.000000,1.000000,1.212594,1.000000,2.949222
8,17.964527,0.471831,4.535254,9.070508,59.425680,1.000003,1.000000,1.673955,1.000000,3.238714
9,24.349696,0.478873,4.613242,9.226483,58.878653,1.001444,1.000000,1.995974,1.000000,5.377324



❌ 当前 grid 中没有 Validation Coverage >= 0.90 的 bandwidth

Maximum Validation Coverage:


,bandwidth_km,coverage,mean_half_width,mean_width,interval_score,median_n_eff,p10_n_eff,p90_n_eff,min_n_eff,max_n_eff
24,2331.631595,0.852113,8.952028,17.904055,34.519053,137.963802,134.335336,139.489549,117.620215,139.784330
22,1269.125041,0.852113,8.889864,17.779728,33.456238,124.172178,101.839174,130.993261,24.392752,132.304491
23,1720.212790,0.852113,8.927656,17.855312,34.106315,133.393815,123.356572,136.836857,74.368289,137.465016
21,936.325074,0.838028,8.774750,17.549499,32.891698,108.232114,72.503752,119.529454,6.870959,122.451365
19,509.648951,0.830986,8.475030,16.950060,33.475208,58.804195,27.114614,77.089906,2.655128,84.114118


RuntimeError: No feasible GeoCP bandwidth. Do NOT choose a sub-0.90 bandwidth. Review grid range first.

In [7]:
# ============================================================
# STEP 7B — GeoCP harmonized weighted quantile
#
# 修正：
# local weighted quantile 与 finite-sample Global CP 对齐
#
# Selection:
# 1. 若存在 Coverage >= 0.90:
#       在 feasible set 中最小 Interval Score
#
# 2. 若不存在:
#       fallback = 全 grid 最小 Interval Score
#
# 这个 fallback 从现在起冻结，
# 后面 GeoSIMCP / Additive-EUC / GCCP-B 全部使用同样规则。
# ============================================================

import math
import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 7B — FORMAL GeoCP / HARMONIZED QUANTILE")
print("=" * 80)

ALPHA = 0.1
TARGET_COVERAGE = 0.90

# ============================================================
# 1. finite-sample weighted quantile level
# ============================================================

n_calib = len(calib_residual)

k_local = math.ceil(
    (n_calib + 1)
    * (1 - ALPHA)
)

k_local = min(
    k_local,
    n_calib
)

TAU_LOCAL = (
    k_local
    / n_calib
)

print("Calibration n =", n_calib)
print("k             =", k_local)
print("tau_local     =", TAU_LOCAL)
print("Global CP q   =", q_cp)

# ============================================================
# 2. Harmonized weighted quantile
# ============================================================

def weighted_quantile_harmonized(
    residuals,
    weights,
    tau=TAU_LOCAL
):
    residuals = np.asarray(
        residuals,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    order = np.argsort(residuals)

    s = residuals[order]
    w = weights[order]

    total = w.sum()

    if (
        not np.isfinite(total)
        or total <= 0
    ):
        raise RuntimeError(
            "Invalid kernel weights"
        )

    cumulative = (
        np.cumsum(w)
        / total
    )

    idx = np.searchsorted(
        cumulative,
        tau,
        side="left"
    )

    idx = min(
        idx,
        len(s) - 1
    )

    return s[idx]


# ============================================================
# 3. Check limiting case
#
# uniform weights should exactly reproduce Global CP q
# ============================================================

q_uniform = weighted_quantile_harmonized(
    calib_residual,
    np.ones(n_calib)
)

print("\nUniform-weight q =", q_uniform)
print("Global CP q      =", q_cp)
print(
    "Difference       =",
    q_uniform - q_cp
)

assert np.isclose(
    q_uniform,
    q_cp
)

print(
    "✅ Uniform weighted quantile "
    "exactly recovers Global CP"
)

# ============================================================
# 4. Rerun GeoCP bandwidth grid
# ============================================================

rows = []
results_by_h_v2 = {}

residuals = np.asarray(
    calib_residual,
    dtype=float
)

val_pred_arr = np.asarray(
    pred_val,
    dtype=float
)

val_true_arr = np.asarray(
    y_val,
    dtype=float
)

for h in H_GEO_GRID:

    q_local = np.zeros(
        len(idx_val)
    )

    neff = np.zeros(
        len(idx_val)
    )

    for j in range(
        len(idx_val)
    ):

        w = stable_gaussian_weights(
            D_geo_val_calib[j],
            h
        )

        q_local[j] = (
            weighted_quantile_harmonized(
                residuals,
                w
            )
        )

        neff[j] = (
            effective_sample_size(w)
        )

    lower_v = (
        val_pred_arr
        - q_local
    )

    upper_v = (
        val_pred_arr
        + q_local
    )

    covered_v = (
        (val_true_arr >= lower_v)
        & (val_true_arr <= upper_v)
    )

    width_v = (
        upper_v - lower_v
    )

    iscore_v = interval_score(
        val_true_arr,
        lower_v,
        upper_v,
        alpha=ALPHA
    )

    row = {
        "bandwidth_km": h,

        "coverage":
            np.mean(covered_v),

        "mean_half_width":
            np.mean(q_local),

        "mean_width":
            np.mean(width_v),

        "interval_score":
            np.mean(iscore_v),

        "median_n_eff":
            np.median(neff),

        "p10_n_eff":
            np.quantile(
                neff,
                0.10
            ),

        "p90_n_eff":
            np.quantile(
                neff,
                0.90
            ),
    }

    rows.append(row)

    results_by_h_v2[h] = {
        "q_local": q_local,
        "lower": lower_v,
        "upper": upper_v,
        "covered": covered_v,
        "width": width_v,
        "interval_score": iscore_v,
        "n_eff": neff,
    }


geocp_grid_v2 = pd.DataFrame(
    rows
)

# ============================================================
# 5. Parameter selection
# ============================================================

feasible = geocp_grid_v2[
    geocp_grid_v2["coverage"]
    >= TARGET_COVERAGE
]

if len(feasible) > 0:

    best_idx = (
        feasible[
            "interval_score"
        ].idxmin()
    )

    selection_rule = (
        "coverage>=0.90_then_min_interval_score"
    )

    coverage_constraint_feasible = True

else:

    best_idx = (
        geocp_grid_v2[
            "interval_score"
        ].idxmin()
    )

    selection_rule = (
        "fallback_global_min_interval_score"
    )

    coverage_constraint_feasible = False


best_geo_v2 = (
    geocp_grid_v2
    .loc[best_idx]
)

best_h_geo = float(
    best_geo_v2[
        "bandwidth_km"
    ]
)

best_geo_result = (
    results_by_h_v2[
        best_h_geo
    ]
)

# ============================================================
# 6. Output
# ============================================================

print("\n" + "=" * 80)
print("GeoCP COVERAGE FEASIBILITY")
print("=" * 80)

print(
    "Coverage >= 0.90 feasible:",
    coverage_constraint_feasible
)

print(
    "Maximum validation coverage:",
    geocp_grid_v2[
        "coverage"
    ].max()
)

print(
    "Selection rule:",
    selection_rule
)

print("\n" + "=" * 80)
print("FORMAL GeoCP — SELECTED VALIDATION PARAMETER")
print("=" * 80)

print(
    f"Bandwidth (km)  = "
    f"{best_h_geo:.6f}"
)

print(
    f"Coverage        = "
    f"{best_geo_v2['coverage']:.6f}"
)

print(
    f"Mean Half-width = "
    f"{best_geo_v2['mean_half_width']:.6f}"
)

print(
    f"Mean Width      = "
    f"{best_geo_v2['mean_width']:.6f}"
)

print(
    f"Interval Score  = "
    f"{best_geo_v2['interval_score']:.6f}"
)

print(
    f"Median n_eff    = "
    f"{best_geo_v2['median_n_eff']:.6f}"
)

# ============================================================
# 7. Compare CP / GeoCP
# ============================================================

comparison_v2 = pd.DataFrame([
    {
        "method": "CP",

        "coverage":
            coverage,

        "mean_width":
            mean_width,

        "interval_score":
            mean_interval_score,

        "median_n_eff":
            n_calib,
    },

    {
        "method": "GeoCP",

        "coverage":
            best_geo_v2[
                "coverage"
            ],

        "mean_width":
            best_geo_v2[
                "mean_width"
            ],

        "interval_score":
            best_geo_v2[
                "interval_score"
            ],

        "median_n_eff":
            best_geo_v2[
                "median_n_eff"
            ],
    }
])

print("\n" + "=" * 80)
print("VALIDATION COMPARISON")
print("=" * 80)

display(comparison_v2)

# ============================================================
# 8. Save final GeoCP Validation result
# ============================================================

geocp_grid_v2.to_csv(
    OUTPUT_DIR
    / "pm25_formal_geocp_validation_grid_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

best_geo_save = pd.DataFrame([{
    **best_geo_v2.to_dict(),

    "tau_local":
        TAU_LOCAL,

    "conformal_k":
        k_local,

    "coverage_constraint_feasible":
        coverage_constraint_feasible,

    "selection_rule":
        selection_rule,
}])

best_geo_save.to_csv(
    OUTPUT_DIR
    / "pm25_formal_geocp_validation_best_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 7B 完成")
print(
    "✅ weighted quantile convention 已冻结"
)
print(
    "✅ parameter fallback rule 已冻结"
)
print(
    "⚠️ Frozen Test y 仍未读取"
)

STEP 7B — FORMAL GeoCP / HARMONIZED QUANTILE
Calibration n = 141
k             = 128
tau_local     = 0.9078014184397163
Global CP q   = 9.133607116712255

Uniform-weight q = 9.133607116712255
Global CP q      = 9.133607116712255
Difference       = 0.0
✅ Uniform weighted quantile exactly recovers Global CP

GeoCP COVERAGE FEASIBILITY
Coverage >= 0.90 feasible: False
Maximum validation coverage: 0.8591549295774648
Selection rule: fallback_global_min_interval_score

FORMAL GeoCP — SELECTED VALIDATION PARAMETER
Bandwidth (km)  = 690.794537
Coverage        = 0.845070
Mean Half-width = 8.872443
Mean Width      = 17.744885
Interval Score  = 32.534134
Median n_eff    = 83.872608

VALIDATION COMPARISON


,method,coverage,mean_width,interval_score,median_n_eff
0,CP,0.859155,18.267214,34.808273,141.000000
1,GeoCP,0.845070,17.744885,32.534134,83.872608



✅ STEP 7B 完成
✅ weighted quantile convention 已冻结
✅ parameter fallback rule 已冻结
⚠️ Frozen Test y 仍未读取


In [8]:
# ============================================================
# STEP 8 — Formal GeoSIMCP-EUC
#
# Original GeoSIMCP joint-distance architecture
# Calibration -> Validation parameter selection
# Frozen Test remains untouched
#
# Common conformal layer:
# uses the harmonized finite-sample weighted quantile from STEP 7B
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

print("=" * 80)
print("STEP 8 — FORMAL GeoSIMCP-EUC / VALIDATION")
print("=" * 80)

ALPHA = 0.1
TARGET_COVERAGE = 0.90

# ============================================================
# 1. Feature space
#
# StandardScaler is fitted on Calibration ONLY.
# Validation is transform only.
# ============================================================

FEATURES = [
    "dem",
    "sp",
    "tp",
    "blh",
    "r",
    "aod_sat",
    "ndvi",
]

feat_calib = (
    data.iloc[idx_calib]
    [FEATURES]
    .to_numpy(dtype=float)
)

feat_val = (
    data.iloc[idx_val]
    [FEATURES]
    .to_numpy(dtype=float)
)

feature_scaler = StandardScaler()

feat_calib_z = feature_scaler.fit_transform(
    feat_calib
)

feat_val_z = feature_scaler.transform(
    feat_val
)

print(
    "Calibration feature shape:",
    feat_calib_z.shape
)

print(
    "Validation feature shape:",
    feat_val_z.shape
)

# ============================================================
# 2. Author-style geographic distance
#
# IMPORTANT:
# Original GeoSIMCP uses raw lon/lat Euclidean distance.
#
# This is intentionally kept for the literature baseline.
# It is NOT the geographic metric used by GCCP-B later.
# ============================================================

geo_calib_degree = (
    data.iloc[idx_calib]
    [["lon", "lat"]]
    .to_numpy(dtype=float)
)

geo_val_degree = (
    data.iloc[idx_val]
    [["lon", "lat"]]
    .to_numpy(dtype=float)
)

D_geo_degree = pairwise_distances(
    geo_val_degree,
    geo_calib_degree,
    metric="euclidean"
)

# ============================================================
# 3. Standardized Euclidean feature distance
# ============================================================

D_feat_euc = pairwise_distances(
    feat_val_z,
    feat_calib_z,
    metric="euclidean"
)

# Match author's small positive convention
D_feat_euc = (
    D_feat_euc
    + 1e-8
)

print(
    "\nGeographic distance matrix:",
    D_geo_degree.shape
)

print(
    "Feature distance matrix:",
    D_feat_euc.shape
)

# ============================================================
# 4. Distance-scale audit
# ============================================================

def distance_stats(x):

    x = np.asarray(
        x,
        dtype=float
    ).ravel()

    return {
        "min": np.min(x),
        "p05": np.quantile(x, 0.05),
        "p25": np.quantile(x, 0.25),
        "median": np.median(x),
        "p75": np.quantile(x, 0.75),
        "p95": np.quantile(x, 0.95),
        "max": np.max(x),
    }


scale_df = pd.DataFrame({
    "geo_degree":
        distance_stats(
            D_geo_degree
        ),

    "feature_EUC_z":
        distance_stats(
            D_feat_euc
        ),
}).T

print("\n" + "=" * 80)
print("GeoSIMCP DISTANCE SCALE")
print("=" * 80)

display(scale_df)

# ============================================================
# 5. Freeze author-aligned parameter grids
#
# Exact executable manuscript grid:
# bandwidth = linspace(0.1, 5.0, 20)
# lambda    = linspace(0.0, 1.0, 20)
# ============================================================

GEOSIM_BW_GRID = np.linspace(
    0.1,
    5.0,
    20
)

LAMBDA_GRID = np.linspace(
    0.0,
    1.0,
    20
)

print("\nBandwidth grid:")
print(
    np.round(
        GEOSIM_BW_GRID,
        6
    )
)

print("\nLambda grid:")
print(
    np.round(
        LAMBDA_GRID,
        6
    )
)

# ============================================================
# 6. Shared Gaussian kernel
#
# d_joint =
# sqrt(lambda*d_geo^2 + (1-lambda)*d_feat^2)
#
# w =
# exp[-0.5*(d_joint/h)^2]
#
# Numerically stable implementation.
# ============================================================

def stable_kernel_from_distance(
    distance,
    bandwidth
):
    d = np.asarray(
        distance,
        dtype=float
    )

    log_w = (
        -0.5
        * (d / bandwidth) ** 2
    )

    log_w -= np.max(
        log_w
    )

    return np.exp(
        log_w
    )


# ============================================================
# 7. Fixed shared inputs
# ============================================================

residuals = np.asarray(
    calib_residual,
    dtype=float
)

val_pred_arr = np.asarray(
    pred_val,
    dtype=float
)

val_true_arr = np.asarray(
    y_val,
    dtype=float
)

# ============================================================
# 8. 20 x 20 grid search
# ============================================================

rows = []

results_geosim_euc = {}

for bw in GEOSIM_BW_GRID:

    for lam in LAMBDA_GRID:

        D_joint = np.sqrt(
            lam
            * D_geo_degree ** 2
            +
            (1.0 - lam)
            * D_feat_euc ** 2
        )

        q_local = np.zeros(
            len(idx_val),
            dtype=float
        )

        neff = np.zeros(
            len(idx_val),
            dtype=float
        )

        for j in range(
            len(idx_val)
        ):

            w = stable_kernel_from_distance(
                D_joint[j],
                bw
            )

            q_local[j] = (
                weighted_quantile_harmonized(
                    residuals,
                    w,
                    tau=TAU_LOCAL
                )
            )

            neff[j] = (
                effective_sample_size(
                    w
                )
            )

        lower_v = (
            val_pred_arr
            - q_local
        )

        upper_v = (
            val_pred_arr
            + q_local
        )

        covered_v = (
            (val_true_arr >= lower_v)
            &
            (val_true_arr <= upper_v)
        )

        width_v = (
            upper_v
            - lower_v
        )

        iscore_v = interval_score(
            val_true_arr,
            lower_v,
            upper_v,
            alpha=ALPHA
        )

        row = {
            "bandwidth":
                bw,

            "lambda":
                lam,

            "coverage":
                np.mean(
                    covered_v
                ),

            "mean_half_width":
                np.mean(
                    q_local
                ),

            "mean_width":
                np.mean(
                    width_v
                ),

            "interval_score":
                np.mean(
                    iscore_v
                ),

            "median_n_eff":
                np.median(
                    neff
                ),

            "p10_n_eff":
                np.quantile(
                    neff,
                    0.10
                ),

            "p90_n_eff":
                np.quantile(
                    neff,
                    0.90
                ),
        }

        rows.append(
            row
        )

        results_geosim_euc[
            (float(bw), float(lam))
        ] = {
            "q_local":
                q_local,

            "lower":
                lower_v,

            "upper":
                upper_v,

            "covered":
                covered_v,

            "width":
                width_v,

            "interval_score":
                iscore_v,

            "n_eff":
                neff,
        }


geosim_euc_grid = pd.DataFrame(
    rows
)

# ============================================================
# 9. Selection rule
#
# Same rule frozen in STEP 7B:
#
# If feasible coverage >= 0.90:
#     minimum Interval Score
#
# Else:
#     global minimum Interval Score
# ============================================================

feasible = geosim_euc_grid[
    geosim_euc_grid["coverage"]
    >= TARGET_COVERAGE
]

if len(feasible) > 0:

    best_idx = (
        feasible[
            "interval_score"
        ].idxmin()
    )

    coverage_constraint_feasible = True

    selection_rule = (
        "coverage>=0.90_then_min_interval_score"
    )

else:

    best_idx = (
        geosim_euc_grid[
            "interval_score"
        ].idxmin()
    )

    coverage_constraint_feasible = False

    selection_rule = (
        "fallback_global_min_interval_score"
    )


best_euc = (
    geosim_euc_grid
    .loc[best_idx]
)

best_bw_euc = float(
    best_euc[
        "bandwidth"
    ]
)

best_lambda_euc = float(
    best_euc[
        "lambda"
    ]
)

best_euc_result = (
    results_geosim_euc[
        (
            best_bw_euc,
            best_lambda_euc
        )
    ]
)

# ============================================================
# 10. Output
# ============================================================

print("\n" + "=" * 80)
print("GeoSIMCP-EUC COVERAGE FEASIBILITY")
print("=" * 80)

print(
    "Coverage >= 0.90 feasible:",
    coverage_constraint_feasible
)

print(
    "Maximum validation coverage:",
    geosim_euc_grid[
        "coverage"
    ].max()
)

print(
    "Selection rule:",
    selection_rule
)

print("\n" + "=" * 80)
print("FORMAL GeoSIMCP-EUC — SELECTED VALIDATION PARAMETER")
print("=" * 80)

print(
    f"Bandwidth       = "
    f"{best_bw_euc:.6f}"
)

print(
    f"Lambda          = "
    f"{best_lambda_euc:.6f}"
)

print(
    f"Coverage        = "
    f"{best_euc['coverage']:.6f}"
)

print(
    f"Mean Half-width = "
    f"{best_euc['mean_half_width']:.6f}"
)

print(
    f"Mean Width      = "
    f"{best_euc['mean_width']:.6f}"
)

print(
    f"Interval Score  = "
    f"{best_euc['interval_score']:.6f}"
)

print(
    f"Median n_eff    = "
    f"{best_euc['median_n_eff']:.6f}"
)

# ============================================================
# 11. Boundary diagnostics
# ============================================================

lambda_at_boundary = (
    np.isclose(
        best_lambda_euc,
        LAMBDA_GRID.min()
    )
    or
    np.isclose(
        best_lambda_euc,
        LAMBDA_GRID.max()
    )
)

bandwidth_at_boundary = (
    np.isclose(
        best_bw_euc,
        GEOSIM_BW_GRID.min()
    )
    or
    np.isclose(
        best_bw_euc,
        GEOSIM_BW_GRID.max()
    )
)

print("\n" + "=" * 80)
print("BOUNDARY DIAGNOSTIC")
print("=" * 80)

print(
    "Lambda at boundary:",
    lambda_at_boundary
)

print(
    "Bandwidth at boundary:",
    bandwidth_at_boundary
)

if np.isclose(
    best_lambda_euc,
    0.0
):
    print(
        "⚠️ λ = 0: selected model "
        "is feature-only"
    )

elif np.isclose(
    best_lambda_euc,
    1.0
):
    print(
        "⚠️ λ = 1: selected model "
        "is geographic-only"
    )

# ============================================================
# 12. Validation comparison
# ============================================================

validation_methods = pd.DataFrame([
    {
        "method": "CP",

        "coverage":
            coverage,

        "mean_width":
            mean_width,

        "interval_score":
            mean_interval_score,

        "median_n_eff":
            n_calib,
    },

    {
        "method": "GeoCP",

        "coverage":
            best_geo_v2[
                "coverage"
            ],

        "mean_width":
            best_geo_v2[
                "mean_width"
            ],

        "interval_score":
            best_geo_v2[
                "interval_score"
            ],

        "median_n_eff":
            best_geo_v2[
                "median_n_eff"
            ],
    },

    {
        "method": "GeoSIMCP-EUC",

        "coverage":
            best_euc[
                "coverage"
            ],

        "mean_width":
            best_euc[
                "mean_width"
            ],

        "interval_score":
            best_euc[
                "interval_score"
            ],

        "median_n_eff":
            best_euc[
                "median_n_eff"
            ],
    }
])

print("\n" + "=" * 80)
print("FORMAL VALIDATION COMPARISON")
print("=" * 80)

display(
    validation_methods
)

# ============================================================
# 13. Save
# ============================================================

geosim_euc_grid.to_csv(
    OUTPUT_DIR
    / "pm25_formal_geosimcp_euc_validation_grid_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    **best_euc.to_dict(),

    "tau_local":
        TAU_LOCAL,

    "coverage_constraint_feasible":
        coverage_constraint_feasible,

    "selection_rule":
        selection_rule,

    "geo_distance":
        "raw_lon_lat_euclidean_degree",

    "feature_distance":
        "standardized_euclidean",

    "feature_scaler_fit":
        "calibration_only",
}]).to_csv(
    OUTPUT_DIR
    / "pm25_formal_geosimcp_euc_validation_best_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

# 保存 scaler 参数，后面可复查
scaler_df = pd.DataFrame({
    "feature": FEATURES,
    "mean": feature_scaler.mean_,
    "scale": feature_scaler.scale_,
})

scaler_df.to_csv(
    OUTPUT_DIR
    / "pm25_formal_feature_scaler_calibration_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 8 完成")
print("✅ GeoSIMCP-EUC Validation 参数已冻结")
print("⚠️ Frozen Test y 仍未读取")

STEP 8 — FORMAL GeoSIMCP-EUC / VALIDATION
Calibration feature shape: (141, 7)
Validation feature shape: (142, 7)

Geographic distance matrix: (142, 141)
Feature distance matrix: (142, 141)

GeoSIMCP DISTANCE SCALE


,min,p05,p25,median,p75,p95,max
geo_degree,0.011007,2.725745,7.003487,11.297420,16.629592,29.463881,55.279616
feature_EUC_z,0.044035,1.119863,2.101337,2.981901,4.180837,6.896668,20.456417



Bandwidth grid:
[0.1      0.357895 0.615789 0.873684 1.131579 1.389474 1.647368 1.905263
 2.163158 2.421053 2.678947 2.936842 3.194737 3.452632 3.710526 3.968421
 4.226316 4.484211 4.742105 5.      ]

Lambda grid:
[0.       0.052632 0.105263 0.157895 0.210526 0.263158 0.315789 0.368421
 0.421053 0.473684 0.526316 0.578947 0.631579 0.684211 0.736842 0.789474
 0.842105 0.894737 0.947368 1.      ]

GeoSIMCP-EUC COVERAGE FEASIBILITY
Coverage >= 0.90 feasible: False
Maximum validation coverage: 0.852112676056338
Selection rule: fallback_global_min_interval_score

FORMAL GeoSIMCP-EUC — SELECTED VALIDATION PARAMETER
Bandwidth       = 1.647368
Lambda          = 0.000000
Coverage        = 0.823944
Mean Half-width = 8.584840
Mean Width      = 17.169679
Interval Score  = 32.012755
Median n_eff    = 85.099852

BOUNDARY DIAGNOSTIC
Lambda at boundary: True
Bandwidth at boundary: False
⚠️ λ = 0: selected model is feature-only

FORMAL VALIDATION COMPARISON


,method,coverage,mean_width,interval_score,median_n_eff
0,CP,0.859155,18.267214,34.808273,141.000000
1,GeoCP,0.845070,17.744885,32.534134,83.872608
2,GeoSIMCP-EUC,0.823944,17.169679,32.012755,85.099852



✅ STEP 8 完成
✅ GeoSIMCP-EUC Validation 参数已冻结
⚠️ Frozen Test y 仍未读取


In [9]:
# ============================================================
# STEP 9 — Formal GeoSIMCP-MND
#
# Author-style MND feature distance
# Author-style raw lon/lat geographic distance
#
# Calibration -> Validation selection only
# Frozen Test remains untouched
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 9 — FORMAL GeoSIMCP-MND / VALIDATION")
print("=" * 80)

ALPHA = 0.1
TARGET_COVERAGE = 0.90

# ============================================================
# 1. Calibration / Validation raw features
# ============================================================

feat_calib_mnd = (
    data.iloc[idx_calib]
    [FEATURES]
    .to_numpy(dtype=float)
)

feat_val_mnd = (
    data.iloc[idx_val]
    [FEATURES]
    .to_numpy(dtype=float)
)

# ============================================================
# 2. Calibration feature ranges
#
# Exactly follows author implementation:
# range_k = max(calib_k) - min(calib_k)
# ============================================================

feature_ranges_mnd = np.ptp(
    feat_calib_mnd,
    axis=0
)

range_df = pd.DataFrame({
    "feature": FEATURES,
    "range": feature_ranges_mnd
})

print("\n" + "=" * 80)
print("MND CALIBRATION FEATURE RANGE")
print("=" * 80)

display(range_df)

zero_mask = np.isclose(
    feature_ranges_mnd,
    0.0
)

if zero_mask.any():
    print("\n⚠️ Zero-range features:")
    print(
        np.array(FEATURES)[
            zero_mask
        ]
    )
else:
    print(
        "\n✅ No zero-range feature "
        "in Formal Calibration"
    )

# Author convention
ranges_safe = (
    feature_ranges_mnd.copy()
)

ranges_safe[
    zero_mask
] = 1e-8

# ============================================================
# 3. MND distance
#
# Author:
#
# similarity_k =
# 1 - |x_test - x_calib| / range_k
#
# similarity = min_k similarity_k
#
# d_feat = 1 - similarity
#
# Algebraically:
#
# d_feat =
# max_k |diff_k| / range_k
# ============================================================

diffs = np.abs(
    feat_val_mnd[:, None, :]
    -
    feat_calib_mnd[None, :, :]
)

scaled_diff = (
    diffs
    /
    ranges_safe[
        None,
        None,
        :
    ]
)

D_feat_mnd = np.max(
    scaled_diff,
    axis=2
)

assert D_feat_mnd.shape == (
    len(idx_val),
    len(idx_calib)
)

# ============================================================
# 4. Geographic distance
#
# Keep original GeoSIMCP literature baseline:
# raw lon/lat Euclidean degrees
#
# D_geo_degree was already frozen in STEP 8
# ============================================================

assert D_geo_degree.shape == (
    len(idx_val),
    len(idx_calib)
)

# ============================================================
# 5. Distance-scale audit
# ============================================================

mnd_scale_df = pd.DataFrame({
    "geo_degree":
        distance_stats(
            D_geo_degree
        ),

    "feature_MND":
        distance_stats(
            D_feat_mnd
        ),
}).T

print("\n" + "=" * 80)
print("GeoSIMCP-MND DISTANCE SCALE")
print("=" * 80)

display(
    mnd_scale_df
)

# ============================================================
# 6. Check Validation values outside Calibration range
#
# MND can be > 1.
# This is allowed by author's formula.
# ============================================================

outside_count = np.sum(
    D_feat_mnd > 1.0
)

outside_fraction = (
    outside_count
    / D_feat_mnd.size
)

print(
    "\nPairs with MND > 1:",
    outside_count,
    "/",
    D_feat_mnd.size
)

print(
    "Fraction:",
    outside_fraction
)

print(
    "Maximum MND:",
    D_feat_mnd.max()
)

# ============================================================
# 7. Use SAME author-aligned grids
# ============================================================

MND_BW_GRID = np.linspace(
    0.1,
    5.0,
    20
)

MND_LAMBDA_GRID = np.linspace(
    0.0,
    1.0,
    20
)

# ============================================================
# 8. Grid search
# ============================================================

residuals = np.asarray(
    calib_residual,
    dtype=float
)

val_pred_arr = np.asarray(
    pred_val,
    dtype=float
)

val_true_arr = np.asarray(
    y_val,
    dtype=float
)

rows = []
results_mnd = {}

for bw in MND_BW_GRID:

    for lam in MND_LAMBDA_GRID:

        D_joint = np.sqrt(
            lam
            * D_geo_degree ** 2
            +
            (1.0 - lam)
            * D_feat_mnd ** 2
        )

        q_local = np.zeros(
            len(idx_val),
            dtype=float
        )

        neff = np.zeros(
            len(idx_val),
            dtype=float
        )

        for j in range(
            len(idx_val)
        ):

            w = stable_kernel_from_distance(
                D_joint[j],
                bw
            )

            q_local[j] = (
                weighted_quantile_harmonized(
                    residuals,
                    w,
                    tau=TAU_LOCAL
                )
            )

            neff[j] = (
                effective_sample_size(
                    w
                )
            )

        lower_v = (
            val_pred_arr
            - q_local
        )

        upper_v = (
            val_pred_arr
            + q_local
        )

        covered_v = (
            (val_true_arr >= lower_v)
            &
            (val_true_arr <= upper_v)
        )

        width_v = (
            upper_v
            - lower_v
        )

        iscore_v = interval_score(
            val_true_arr,
            lower_v,
            upper_v,
            alpha=ALPHA
        )

        row = {
            "bandwidth":
                bw,

            "lambda":
                lam,

            "coverage":
                np.mean(
                    covered_v
                ),

            "mean_half_width":
                np.mean(
                    q_local
                ),

            "mean_width":
                np.mean(
                    width_v
                ),

            "interval_score":
                np.mean(
                    iscore_v
                ),

            "median_n_eff":
                np.median(
                    neff
                ),

            "p10_n_eff":
                np.quantile(
                    neff,
                    0.10
                ),

            "p90_n_eff":
                np.quantile(
                    neff,
                    0.90
                ),
        }

        rows.append(
            row
        )

        results_mnd[
            (float(bw), float(lam))
        ] = {
            "q_local":
                q_local,

            "lower":
                lower_v,

            "upper":
                upper_v,

            "covered":
                covered_v,

            "width":
                width_v,

            "interval_score":
                iscore_v,

            "n_eff":
                neff,
        }


mnd_grid_df = pd.DataFrame(
    rows
)

# ============================================================
# 9. Same frozen selection rule
# ============================================================

feasible = mnd_grid_df[
    mnd_grid_df["coverage"]
    >= TARGET_COVERAGE
]

if len(feasible) > 0:

    best_idx = (
        feasible[
            "interval_score"
        ].idxmin()
    )

    coverage_constraint_feasible = True

    selection_rule = (
        "coverage>=0.90_then_min_interval_score"
    )

else:

    best_idx = (
        mnd_grid_df[
            "interval_score"
        ].idxmin()
    )

    coverage_constraint_feasible = False

    selection_rule = (
        "fallback_global_min_interval_score"
    )


best_mnd = (
    mnd_grid_df
    .loc[best_idx]
)

best_bw_mnd = float(
    best_mnd[
        "bandwidth"
    ]
)

best_lambda_mnd = float(
    best_mnd[
        "lambda"
    ]
)

best_mnd_result = (
    results_mnd[
        (
            best_bw_mnd,
            best_lambda_mnd
        )
    ]
)

# ============================================================
# 10. Output
# ============================================================

print("\n" + "=" * 80)
print("GeoSIMCP-MND COVERAGE FEASIBILITY")
print("=" * 80)

print(
    "Coverage >= 0.90 feasible:",
    coverage_constraint_feasible
)

print(
    "Maximum validation coverage:",
    mnd_grid_df[
        "coverage"
    ].max()
)

print(
    "Selection rule:",
    selection_rule
)

print("\n" + "=" * 80)
print("FORMAL GeoSIMCP-MND — SELECTED VALIDATION PARAMETER")
print("=" * 80)

print(
    f"Bandwidth       = "
    f"{best_bw_mnd:.6f}"
)

print(
    f"Lambda          = "
    f"{best_lambda_mnd:.6f}"
)

print(
    f"Coverage        = "
    f"{best_mnd['coverage']:.6f}"
)

print(
    f"Mean Half-width = "
    f"{best_mnd['mean_half_width']:.6f}"
)

print(
    f"Mean Width      = "
    f"{best_mnd['mean_width']:.6f}"
)

print(
    f"Interval Score  = "
    f"{best_mnd['interval_score']:.6f}"
)

print(
    f"Median n_eff    = "
    f"{best_mnd['median_n_eff']:.6f}"
)

# ============================================================
# 11. Boundary diagnostic
# ============================================================

lambda_boundary_mnd = (
    np.isclose(
        best_lambda_mnd,
        MND_LAMBDA_GRID.min()
    )
    or
    np.isclose(
        best_lambda_mnd,
        MND_LAMBDA_GRID.max()
    )
)

bw_boundary_mnd = (
    np.isclose(
        best_bw_mnd,
        MND_BW_GRID.min()
    )
    or
    np.isclose(
        best_bw_mnd,
        MND_BW_GRID.max()
    )
)

print("\n" + "=" * 80)
print("MND BOUNDARY DIAGNOSTIC")
print("=" * 80)

print(
    "Lambda at boundary:",
    lambda_boundary_mnd
)

print(
    "Bandwidth at boundary:",
    bw_boundary_mnd
)

if np.isclose(
    best_lambda_mnd,
    0.0
):
    print(
        "⚠️ λ = 0: MND selected "
        "feature-only"
    )

elif np.isclose(
    best_lambda_mnd,
    1.0
):
    print(
        "⚠️ λ = 1: MND selected "
        "geographic-only"
    )

# ============================================================
# 12. Validation comparison
# ============================================================

validation_baselines = pd.DataFrame([
    {
        "method": "CP",
        "coverage": coverage,
        "mean_width": mean_width,
        "interval_score":
            mean_interval_score,
        "median_n_eff":
            n_calib,
    },

    {
        "method": "GeoCP",
        "coverage":
            best_geo_v2["coverage"],
        "mean_width":
            best_geo_v2["mean_width"],
        "interval_score":
            best_geo_v2[
                "interval_score"
            ],
        "median_n_eff":
            best_geo_v2[
                "median_n_eff"
            ],
    },

    {
        "method": "GeoSIMCP-EUC",
        "coverage":
            best_euc["coverage"],
        "mean_width":
            best_euc["mean_width"],
        "interval_score":
            best_euc[
                "interval_score"
            ],
        "median_n_eff":
            best_euc[
                "median_n_eff"
            ],
    },

    {
        "method": "GeoSIMCP-MND",
        "coverage":
            best_mnd["coverage"],
        "mean_width":
            best_mnd["mean_width"],
        "interval_score":
            best_mnd[
                "interval_score"
            ],
        "median_n_eff":
            best_mnd[
                "median_n_eff"
            ],
    },
])

print("\n" + "=" * 80)
print("FORMAL BASELINE VALIDATION SUMMARY")
print("=" * 80)

display(
    validation_baselines
)

# ============================================================
# 13. Save
# ============================================================

mnd_grid_df.to_csv(
    OUTPUT_DIR
    / "pm25_formal_geosimcp_mnd_validation_grid_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    **best_mnd.to_dict(),

    "tau_local":
        TAU_LOCAL,

    "coverage_constraint_feasible":
        coverage_constraint_feasible,

    "selection_rule":
        selection_rule,

    "geo_distance":
        "raw_lon_lat_euclidean_degree",

    "feature_distance":
        "MND_calibration_range",

    "range_fit":
        "calibration_only",
}]).to_csv(
    OUTPUT_DIR
    / "pm25_formal_geosimcp_mnd_validation_best_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

range_df.to_csv(
    OUTPUT_DIR
    / "pm25_formal_mnd_calibration_ranges_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 9 完成")
print("✅ 文献 baseline 已完成到 MND")
print("⚠️ Frozen Test y 仍未读取")

STEP 9 — FORMAL GeoSIMCP-MND / VALIDATION

MND CALIBRATION FEATURE RANGE


,feature,range
0,dem,4519.000000
1,sp,43791.054688
2,tp,0.006368
3,blh,962.343307
4,r,71.300009
5,aod_sat,0.788014
6,ndvi,5677.000000



✅ No zero-range feature in Formal Calibration

GeoSIMCP-MND DISTANCE SCALE


,min,p05,p25,median,p75,p95,max
geo_degree,0.011007,2.725745,7.003487,11.297420,16.629592,29.463881,55.279616
feature_MND,0.006144,0.150273,0.292295,0.419088,0.574614,0.820189,2.303982



Pairs with MND > 1: 187 / 20022
Fraction: 0.009339726301068824
Maximum MND: 2.303982450344057

GeoSIMCP-MND COVERAGE FEASIBILITY
Coverage >= 0.90 feasible: False
Maximum validation coverage: 0.8591549295774648
Selection rule: fallback_global_min_interval_score

FORMAL GeoSIMCP-MND — SELECTED VALIDATION PARAMETER
Bandwidth       = 2.421053
Lambda          = 0.157895
Coverage        = 0.845070
Mean Half-width = 8.803340
Mean Width      = 17.606680
Interval Score  = 32.446124
Median n_eff    = 75.242736

MND BOUNDARY DIAGNOSTIC
Lambda at boundary: False
Bandwidth at boundary: False

FORMAL BASELINE VALIDATION SUMMARY


,method,coverage,mean_width,interval_score,median_n_eff
0,CP,0.859155,18.267214,34.808273,141.000000
1,GeoCP,0.845070,17.744885,32.534134,83.872608
2,GeoSIMCP-EUC,0.823944,17.169679,32.012755,85.099852
3,GeoSIMCP-MND,0.845070,17.606680,32.446124,75.242736



✅ STEP 9 完成
✅ 文献 baseline 已完成到 MND
⚠️ Frozen Test y 仍未读取


In [10]:
# ============================================================
# STEP 10 — Additive-EUC controlled baseline
#
# Geographic:
#   Haversine km + h_geo
#
# Feature:
#   standardized Euclidean + h_feat
#
# Fusion:
#   w = lambda*K_geo + (1-lambda)*K_feat
#
# Validation ONLY
# Frozen Test remains untouched
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 10 — ADDITIVE-EUC / VALIDATION")
print("=" * 80)

ALPHA = 0.1
TARGET_COVERAGE = 0.90

# ============================================================
# 1. Distances already available
#
# D_geo_val_calib : Haversine km, STEP 7
# D_feat_euc      : standardized feature Euclidean, STEP 8
# ============================================================

assert D_geo_val_calib.shape == (
    len(idx_val),
    len(idx_calib)
)

assert D_feat_euc.shape == (
    len(idx_val),
    len(idx_calib)
)

# ============================================================
# 2. Geographic bandwidth grid
#
# Reuse EXACTLY the grid frozen in STEP 7
# ============================================================

H_GEO_ADDITIVE_GRID = np.asarray(
    H_GEO_GRID,
    dtype=float
)

# ============================================================
# 3. Feature bandwidth grid
#
# Constructed ONLY from feature distances,
# without using Validation y.
# ============================================================

nearest_feat = D_feat_euc.min(
    axis=1
)

all_feat_dist = D_feat_euc.ravel()

h_feat_min = max(
    0.05,
    0.5 * np.quantile(
        nearest_feat,
        0.05
    )
)

h_feat_max = np.quantile(
    all_feat_dist,
    0.90
)

H_FEAT_GRID = np.geomspace(
    h_feat_min,
    h_feat_max,
    20
)

# ============================================================
# 4. Lambda grid
#
# Freeze this grid now.
# GCCP-B later MUST use exactly the same lambda candidates.
# ============================================================

LAMBDA_ADDITIVE_GRID = np.linspace(
    0.0,
    1.0,
    11
)

print("\n" + "=" * 80)
print("FROZEN ADDITIVE PARAMETER GRIDS")
print("=" * 80)

print(
    "h_geo candidates:",
    len(H_GEO_ADDITIVE_GRID)
)

print(
    "h_geo range:",
    H_GEO_ADDITIVE_GRID.min(),
    "→",
    H_GEO_ADDITIVE_GRID.max(),
    "km"
)

print(
    "\nh_feat candidates:",
    len(H_FEAT_GRID)
)

print(
    "h_feat range:",
    h_feat_min,
    "→",
    h_feat_max
)

print(
    "\nlambda:",
    LAMBDA_ADDITIVE_GRID
)

# ============================================================
# 5. Raw Gaussian kernel
#
# IMPORTANT:
# For additive fusion we CANNOT independently max-shift
# K_geo and K_feat, because that would alter lambda meaning.
#
# So we use the actual Gaussian kernel.
# Clip only at machine-safe log scale.
# ============================================================

def raw_gaussian_kernel(
    D,
    h
):
    log_w = (
        -0.5
        * (D / h) ** 2
    )

    # Numerical floor only.
    # exp(-700) ~ 1e-304
    log_w = np.clip(
        log_w,
        -700,
        0
    )

    return np.exp(
        log_w
    )

# ============================================================
# 6. Vectorized weighted quantile
#
# residual ordering is identical for every Validation point,
# so sort once.
# ============================================================

residuals = np.asarray(
    calib_residual,
    dtype=float
)

resid_order = np.argsort(
    residuals
)

resid_sorted = residuals[
    resid_order
]


def weighted_q_matrix(
    W,
    tau=TAU_LOCAL
):
    """
    W: n_validation x n_calibration
    """

    W = np.asarray(
        W,
        dtype=float
    )

    W_sorted = W[
        :,
        resid_order
    ]

    row_sum = W_sorted.sum(
        axis=1,
        keepdims=True
    )

    if np.any(
        (~np.isfinite(row_sum))
        |
        (row_sum <= 0)
    ):
        raise RuntimeError(
            "Invalid additive kernel weights"
        )

    cum = np.cumsum(
        W_sorted,
        axis=1
    ) / row_sum

    first_idx = np.argmax(
        cum >= tau,
        axis=1
    )

    return resid_sorted[
        first_idx
    ]


def neff_matrix(W):

    numerator = (
        W.sum(axis=1) ** 2
    )

    denominator = (
        (W ** 2).sum(axis=1)
    )

    return (
        numerator
        / denominator
    )

# ============================================================
# 7. Precompute kernels
# ============================================================

geo_kernels = {
    float(h): raw_gaussian_kernel(
        D_geo_val_calib,
        h
    )
    for h in H_GEO_ADDITIVE_GRID
}

feat_kernels = {
    float(h): raw_gaussian_kernel(
        D_feat_euc,
        h
    )
    for h in H_FEAT_GRID
}

print("\n✅ Geographic / feature kernels precomputed")

# ============================================================
# 8. Grid search
# ============================================================

val_pred_arr = np.asarray(
    pred_val,
    dtype=float
)

val_true_arr = np.asarray(
    y_val,
    dtype=float
)

rows = []

best_payloads = {}

for h_geo in H_GEO_ADDITIVE_GRID:

    K_geo = geo_kernels[
        float(h_geo)
    ]

    for h_feat in H_FEAT_GRID:

        K_feat = feat_kernels[
            float(h_feat)
        ]

        for lam in LAMBDA_ADDITIVE_GRID:

            W = (
                lam * K_geo
                +
                (1.0 - lam)
                * K_feat
            )

            q_local = weighted_q_matrix(
                W,
                tau=TAU_LOCAL
            )

            neff = neff_matrix(
                W
            )

            lower_v = (
                val_pred_arr
                - q_local
            )

            upper_v = (
                val_pred_arr
                + q_local
            )

            covered_v = (
                (val_true_arr >= lower_v)
                &
                (val_true_arr <= upper_v)
            )

            width_v = (
                upper_v
                - lower_v
            )

            iscore_v = interval_score(
                val_true_arr,
                lower_v,
                upper_v,
                alpha=ALPHA
            )

            row = {
                "h_geo_km":
                    float(h_geo),

                "h_feat":
                    float(h_feat),

                "lambda":
                    float(lam),

                "coverage":
                    np.mean(
                        covered_v
                    ),

                "mean_half_width":
                    np.mean(
                        q_local
                    ),

                "mean_width":
                    np.mean(
                        width_v
                    ),

                "interval_score":
                    np.mean(
                        iscore_v
                    ),

                "median_n_eff":
                    np.median(
                        neff
                    ),

                "p10_n_eff":
                    np.quantile(
                        neff,
                        0.10
                    ),

                "p90_n_eff":
                    np.quantile(
                        neff,
                        0.90
                    ),
            }

            rows.append(
                row
            )


additive_grid_df = pd.DataFrame(
    rows
)

print(
    "\nGrid combinations:",
    len(additive_grid_df)
)

# ============================================================
# 9. Same frozen parameter-selection rule
# ============================================================

feasible = additive_grid_df[
    additive_grid_df["coverage"]
    >= TARGET_COVERAGE
]

if len(feasible) > 0:

    best_idx = feasible[
        "interval_score"
    ].idxmin()

    coverage_constraint_feasible = True

    selection_rule = (
        "coverage>=0.90_then_min_interval_score"
    )

else:

    best_idx = additive_grid_df[
        "interval_score"
    ].idxmin()

    coverage_constraint_feasible = False

    selection_rule = (
        "fallback_global_min_interval_score"
    )


best_additive = (
    additive_grid_df
    .loc[best_idx]
)

best_h_geo_add = float(
    best_additive[
        "h_geo_km"
    ]
)

best_h_feat_add = float(
    best_additive[
        "h_feat"
    ]
)

best_lambda_add = float(
    best_additive[
        "lambda"
    ]
)

# ============================================================
# 10. Reconstruct selected result
# ============================================================

K_geo_best = geo_kernels[
    best_h_geo_add
]

K_feat_best = feat_kernels[
    best_h_feat_add
]

W_best = (
    best_lambda_add
    * K_geo_best
    +
    (1 - best_lambda_add)
    * K_feat_best
)

q_best = weighted_q_matrix(
    W_best
)

neff_best = neff_matrix(
    W_best
)

lower_best = (
    val_pred_arr
    - q_best
)

upper_best = (
    val_pred_arr
    + q_best
)

covered_best = (
    (val_true_arr >= lower_best)
    &
    (val_true_arr <= upper_best)
)

width_best = (
    upper_best
    - lower_best
)

iscore_best = interval_score(
    val_true_arr,
    lower_best,
    upper_best,
    alpha=ALPHA
)

# ============================================================
# 11. Output
# ============================================================

print("\n" + "=" * 80)
print("ADDITIVE-EUC COVERAGE FEASIBILITY")
print("=" * 80)

print(
    "Coverage >= 0.90 feasible:",
    coverage_constraint_feasible
)

print(
    "Maximum validation coverage:",
    additive_grid_df[
        "coverage"
    ].max()
)

print(
    "Selection rule:",
    selection_rule
)

print("\n" + "=" * 80)
print("FORMAL ADDITIVE-EUC — SELECTED VALIDATION PARAMETER")
print("=" * 80)

print(
    f"h_geo (km)     = "
    f"{best_h_geo_add:.6f}"
)

print(
    f"h_feat         = "
    f"{best_h_feat_add:.6f}"
)

print(
    f"Lambda         = "
    f"{best_lambda_add:.6f}"
)

print(
    f"Coverage       = "
    f"{best_additive['coverage']:.6f}"
)

print(
    f"Mean Width     = "
    f"{best_additive['mean_width']:.6f}"
)

print(
    f"Interval Score = "
    f"{best_additive['interval_score']:.6f}"
)

print(
    f"Median n_eff   = "
    f"{best_additive['median_n_eff']:.6f}"
)

# ============================================================
# 12. Boundary checks
# ============================================================

print("\n" + "=" * 80)
print("ADDITIVE BOUNDARY DIAGNOSTIC")
print("=" * 80)

print(
    "lambda boundary:",
    (
        np.isclose(
            best_lambda_add,
            0.0
        )
        or
        np.isclose(
            best_lambda_add,
            1.0
        )
    )
)

print(
    "h_geo boundary:",
    (
        np.isclose(
            best_h_geo_add,
            H_GEO_ADDITIVE_GRID.min()
        )
        or
        np.isclose(
            best_h_geo_add,
            H_GEO_ADDITIVE_GRID.max()
        )
    )
)

print(
    "h_feat boundary:",
    (
        np.isclose(
            best_h_feat_add,
            H_FEAT_GRID.min()
        )
        or
        np.isclose(
            best_h_feat_add,
            H_FEAT_GRID.max()
        )
    )
)

# ============================================================
# 13. Controlled comparison
# ============================================================

controlled_comparison = pd.DataFrame([
    {
        "method":
            "GeoSIMCP-EUC",

        "coverage":
            best_euc[
                "coverage"
            ],

        "mean_width":
            best_euc[
                "mean_width"
            ],

        "interval_score":
            best_euc[
                "interval_score"
            ],

        "median_n_eff":
            best_euc[
                "median_n_eff"
            ],
    },

    {
        "method":
            "Additive-EUC",

        "coverage":
            best_additive[
                "coverage"
            ],

        "mean_width":
            best_additive[
                "mean_width"
            ],

        "interval_score":
            best_additive[
                "interval_score"
            ],

        "median_n_eff":
            best_additive[
                "median_n_eff"
            ],
    },
])

print("\n" + "=" * 80)
print("CONTROLLED ARCHITECTURE COMPARISON")
print("=" * 80)

display(
    controlled_comparison
)

# ============================================================
# 14. Save
# ============================================================

additive_grid_df.to_csv(
    OUTPUT_DIR
    / "pm25_formal_additive_euc_validation_grid_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    **best_additive.to_dict(),

    "tau_local":
        TAU_LOCAL,

    "coverage_constraint_feasible":
        coverage_constraint_feasible,

    "selection_rule":
        selection_rule,

    "geo_distance":
        "Haversine_km",

    "feature_distance":
        "standardized_Euclidean",

    "fusion":
        "lambda*K_geo+(1-lambda)*K_feat",
}]).to_csv(
    OUTPUT_DIR
    / "pm25_formal_additive_euc_validation_best_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "h_feat":
        H_FEAT_GRID
}).to_csv(
    OUTPUT_DIR
    / "pm25_formal_h_feat_grid_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "lambda":
        LAMBDA_ADDITIVE_GRID
}).to_csv(
    OUTPUT_DIR
    / "pm25_formal_lambda_grid_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 10 完成")
print("✅ Additive architecture 参数规则已冻结")
print("✅ GCCP-B 后续必须复用同一个 lambda grid")
print("⚠️ Frozen Test y 仍未读取")

STEP 10 — ADDITIVE-EUC / VALIDATION

FROZEN ADDITIVE PARAMETER GRIDS
h_geo candidates: 25
h_geo range: 1.5768620734001186 → 2331.6315954622364 km

h_feat candidates: 20
h_feat range: 0.11319637680000501 → 5.623620199428191

lambda: [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]

✅ Geographic / feature kernels precomputed


C:\Users\86186\AppData\Local\Temp\ipykernel_29976\3715966262.py:245: RuntimeWarning: invalid value encountered in divide
  numerator



Grid combinations: 5500

ADDITIVE-EUC COVERAGE FEASIBILITY
Coverage >= 0.90 feasible: False
Maximum validation coverage: 0.8661971830985915
Selection rule: fallback_global_min_interval_score

FORMAL ADDITIVE-EUC — SELECTED VALIDATION PARAMETER
h_geo (km)     = 150.994142
h_feat         = 1.638246
Lambda         = 0.200000
Coverage       = 0.823944
Mean Width     = 17.027888
Interval Score = 31.899938
Median n_eff   = 83.253205

ADDITIVE BOUNDARY DIAGNOSTIC
lambda boundary: False
h_geo boundary: False
h_feat boundary: False

CONTROLLED ARCHITECTURE COMPARISON


,method,coverage,mean_width,interval_score,median_n_eff
0,GeoSIMCP-EUC,0.823944,17.169679,32.012755,85.099852
1,Additive-EUC,0.823944,17.027888,31.899938,83.253205



✅ STEP 10 完成
✅ Additive architecture 参数规则已冻结
✅ GCCP-B 后续必须复用同一个 lambda grid
⚠️ Frozen Test y 仍未读取


In [11]:
# ============================================================
# STEP 10B — Additive-EUC sanity check
#
# 目的：
# 1. 修复 n_eff numerical underflow
# 2. 验证 lambda=0 时结果与 h_geo 完全无关
# 3. 验证 lambda=0 Additive-EUC 与 feature-only
#    GeoSIMCP-EUC 数学上等价
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 10B — ADDITIVE-EUC SANITY CHECK")
print("=" * 80)

# ============================================================
# 1. Stable effective sample size
#
# n_eff 对所有权重乘相同常数不变，
# 所以先除以每一行最大权重，避免平方下溢。
# ============================================================

def neff_matrix_stable(W):

    W = np.asarray(
        W,
        dtype=float
    )

    row_max = np.max(
        W,
        axis=1,
        keepdims=True
    )

    if np.any(
        (~np.isfinite(row_max))
        |
        (row_max <= 0)
    ):
        raise RuntimeError(
            "Invalid kernel weights in n_eff"
        )

    W_scaled = (
        W / row_max
    )

    numerator = (
        W_scaled.sum(axis=1) ** 2
    )

    denominator = (
        (W_scaled ** 2)
        .sum(axis=1)
    )

    return (
        numerator
        / denominator
    )


# ============================================================
# 2. lambda=0 时：
#
# W = K_feat
#
# 因此 h_geo 必须完全不起作用。
# ============================================================

lambda0_rows = additive_grid_df[
    np.isclose(
        additive_grid_df["lambda"],
        0.0
    )
].copy()

# 对每个 h_feat，
# 查看不同 h_geo 下指标最大值 - 最小值
lambda0_invariance = (
    lambda0_rows
    .groupby("h_feat")
    .agg(
        coverage_min=("coverage", "min"),
        coverage_max=("coverage", "max"),

        width_min=("mean_width", "min"),
        width_max=("mean_width", "max"),

        score_min=("interval_score", "min"),
        score_max=("interval_score", "max"),
    )
    .reset_index()
)

lambda0_invariance[
    "coverage_spread"
] = (
    lambda0_invariance["coverage_max"]
    -
    lambda0_invariance["coverage_min"]
)

lambda0_invariance[
    "width_spread"
] = (
    lambda0_invariance["width_max"]
    -
    lambda0_invariance["width_min"]
)

lambda0_invariance[
    "score_spread"
] = (
    lambda0_invariance["score_max"]
    -
    lambda0_invariance["score_min"]
)

print("\n" + "=" * 80)
print("LAMBDA=0 INVARIANCE CHECK")
print("=" * 80)

display(
    lambda0_invariance[
        [
            "h_feat",
            "coverage_spread",
            "width_spread",
            "score_spread"
        ]
    ]
)

print("\nMaximum spreads:")

print(
    "coverage:",
    lambda0_invariance[
        "coverage_spread"
    ].max()
)

print(
    "width   :",
    lambda0_invariance[
        "width_spread"
    ].max()
)

print(
    "score   :",
    lambda0_invariance[
        "score_spread"
    ].max()
)

assert np.allclose(
    lambda0_invariance[
        "coverage_spread"
    ],
    0.0,
    atol=1e-12
)

assert np.allclose(
    lambda0_invariance[
        "width_spread"
    ],
    0.0,
    atol=1e-12
)

assert np.allclose(
    lambda0_invariance[
        "score_spread"
    ],
    0.0,
    atol=1e-12
)

print(
    "\n✅ lambda=0 时结果与 h_geo 完全无关"
)


# ============================================================
# 3. 重新计算 selected Additive-EUC 的稳定 n_eff
# ============================================================

K_feat_selected = raw_gaussian_kernel(
    D_feat_euc,
    best_h_feat_add
)

# lambda=0，所以 W 就是 K_feat
W_selected = K_feat_selected.copy()

neff_selected_stable = (
    neff_matrix_stable(
        W_selected
    )
)

print("\n" + "=" * 80)
print("STABLE n_eff")
print("=" * 80)

print(
    "median n_eff =",
    np.median(
        neff_selected_stable
    )
)

print(
    "p10 n_eff    =",
    np.quantile(
        neff_selected_stable,
        0.10
    )
)

print(
    "p90 n_eff    =",
    np.quantile(
        neff_selected_stable,
        0.90
    )
)

print(
    "min n_eff    =",
    np.min(
        neff_selected_stable
    )
)

print(
    "max n_eff    =",
    np.max(
        neff_selected_stable
    )
)

assert np.all(
    np.isfinite(
        neff_selected_stable
    )
)

print(
    "✅ n_eff 全部 finite"
)


# ============================================================
# 4. Direct equivalence check
#
# 用 GeoSIMCP-EUC 最优 bandwidth，
# 强制 Additive-EUC lambda=0。
#
# 理论上应该和 STEP 8 GeoSIMCP-EUC
# 得到完全一致的 q / Coverage / Width / Score。
# ============================================================

K_feat_geosim_bw = raw_gaussian_kernel(
    D_feat_euc,
    best_bw_euc
)

q_equiv = weighted_q_matrix(
    K_feat_geosim_bw,
    tau=TAU_LOCAL
)

lower_equiv = (
    val_pred_arr
    - q_equiv
)

upper_equiv = (
    val_pred_arr
    + q_equiv
)

covered_equiv = (
    (val_true_arr >= lower_equiv)
    &
    (val_true_arr <= upper_equiv)
)

width_equiv = (
    upper_equiv
    - lower_equiv
)

score_equiv = interval_score(
    val_true_arr,
    lower_equiv,
    upper_equiv,
    alpha=ALPHA
)

equiv_coverage = np.mean(
    covered_equiv
)

equiv_width = np.mean(
    width_equiv
)

equiv_score = np.mean(
    score_equiv
)

print("\n" + "=" * 80)
print("GeoSIMCP-EUC vs ADDITIVE lambda=0 EQUIVALENCE")
print("=" * 80)

equiv_df = pd.DataFrame([
    {
        "method":
            "GeoSIMCP-EUC",

        "bandwidth":
            best_bw_euc,

        "coverage":
            best_euc["coverage"],

        "mean_width":
            best_euc["mean_width"],

        "interval_score":
            best_euc["interval_score"],
    },

    {
        "method":
            "Additive-EUC lambda=0 @ same h",

        "bandwidth":
            best_bw_euc,

        "coverage":
            equiv_coverage,

        "mean_width":
            equiv_width,

        "interval_score":
            equiv_score,
    }
])

display(
    equiv_df
)

print(
    "\nΔCoverage =",
    equiv_coverage
    - best_euc["coverage"]
)

print(
    "ΔWidth    =",
    equiv_width
    - best_euc["mean_width"]
)

print(
    "ΔScore    =",
    equiv_score
    - best_euc["interval_score"]
)

assert np.isclose(
    equiv_coverage,
    best_euc["coverage"],
    atol=1e-12
)

assert np.isclose(
    equiv_width,
    best_euc["mean_width"],
    atol=1e-12
)

assert np.isclose(
    equiv_score,
    best_euc["interval_score"],
    atol=1e-12
)

print(
    "\n✅ lambda=0 下两种架构严格等价"
)


# ============================================================
# 5. Canonical Additive-EUC summary
#
# lambda=0 时 h_geo 不可识别，
# 正式表中应该记录 NA，而不是某个任意 h_geo。
# ============================================================

additive_canonical = pd.DataFrame([{
    "method":
        "Additive-EUC",

    "h_geo_km":
        np.nan,

    "h_feat":
        best_h_feat_add,

    "lambda":
        best_lambda_add,

    "coverage":
        best_additive["coverage"],

    "mean_width":
        best_additive["mean_width"],

    "interval_score":
        best_additive[
            "interval_score"
        ],

    "median_n_eff":
        np.median(
            neff_selected_stable
        ),

    "coverage_constraint_feasible":
        coverage_constraint_feasible,

    "selection_rule":
        selection_rule,

    "boundary_note":
        (
            "lambda=0; geographic "
            "kernel inactive; h_geo not identifiable"
        )
}])

print("\n" + "=" * 80)
print("CANONICAL ADDITIVE-EUC RESULT")
print("=" * 80)

display(
    additive_canonical
)

additive_canonical.to_csv(
    OUTPUT_DIR
    / "pm25_formal_additive_euc_validation_best_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

lambda0_invariance.to_csv(
    OUTPUT_DIR
    / "pm25_formal_additive_lambda0_invariance_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 10B 完成")
print("✅ n_eff 数值问题已修复")
print("✅ Additive λ=0 退化性质已验证")
print("⚠️ Frozen Test y 仍未读取")

STEP 10B — ADDITIVE-EUC SANITY CHECK

LAMBDA=0 INVARIANCE CHECK


,h_feat,coverage_spread,width_spread,score_spread
0,0.113196,0.0,0.0,0.0
1,0.139029,0.0,0.0,0.0
2,0.170757,0.0,0.0,0.0
3,0.209725,0.0,0.0,0.0
4,0.257587,0.0,0.0,0.0
5,0.316371,0.0,0.0,0.0
6,0.388570,0.0,0.0,0.0
7,0.477246,0.0,0.0,0.0
8,0.586158,0.0,0.0,0.0
9,0.719926,0.0,0.0,0.0



Maximum spreads:
coverage: 0.0
width   : 0.0
score   : 0.0

✅ lambda=0 时结果与 h_geo 完全无关

STABLE n_eff
median n_eff = 84.67401342662878
p10 n_eff    = 46.01466961036213
p90 n_eff    = 97.6496927748098
min n_eff    = 1.030620544900306
max n_eff    = 105.2619448010917
✅ n_eff 全部 finite

GeoSIMCP-EUC vs ADDITIVE lambda=0 EQUIVALENCE


,method,bandwidth,coverage,mean_width,interval_score
0,GeoSIMCP-EUC,1.647368,0.823944,17.169679,32.012755
1,Additive-EUC lambda=0 @ same h,1.647368,0.823944,17.169679,32.012755



ΔCoverage = 0.0
ΔWidth    = 0.0
ΔScore    = 0.0

✅ lambda=0 下两种架构严格等价

CANONICAL ADDITIVE-EUC RESULT


,method,h_geo_km,h_feat,lambda,coverage,mean_width,interval_score,median_n_eff,coverage_constraint_feasible,selection_rule,boundary_note
0,Additive-EUC,NaN,1.638246,0.2,0.823944,17.027888,31.899938,84.674013,False,fallback_global_min_interval_score,lambda=0; geographic kernel inactive; h_geo no...



✅ STEP 10B 完成
✅ n_eff 数值问题已修复
✅ Additive λ=0 退化性质已验证
⚠️ Frozen Test y 仍未读取


In [12]:
# ============================================================
# STEP 10C — Correct selected Additive-EUC diagnostics
#
# 修正 STEP 10B：
# 实际 selected lambda = 0.2，不是 0
#
# 因此：
# W = lambda*K_geo + (1-lambda)*K_feat
# h_geo 必须保留
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 10C — CORRECT ADDITIVE-EUC SELECTED RESULT")
print("=" * 80)

print("Selected parameters from STEP 10:")
print("h_geo  =", best_h_geo_add)
print("h_feat =", best_h_feat_add)
print("lambda =", best_lambda_add)

assert best_lambda_add > 0
assert best_lambda_add < 1

print(
    "\n✅ Selected Additive-EUC is a genuine "
    "geo + feature fusion solution"
)

# ============================================================
# 1. Reconstruct ACTUAL selected kernels
# ============================================================

K_geo_selected = raw_gaussian_kernel(
    D_geo_val_calib,
    best_h_geo_add
)

K_feat_selected = raw_gaussian_kernel(
    D_feat_euc,
    best_h_feat_add
)

W_selected = (
    best_lambda_add
    * K_geo_selected
    +
    (1.0 - best_lambda_add)
    * K_feat_selected
)

# ============================================================
# 2. Stable n_eff
# ============================================================

neff_selected = neff_matrix_stable(
    W_selected
)

print("\n" + "=" * 80)
print("SELECTED ADDITIVE n_eff")
print("=" * 80)

print(
    "median n_eff =",
    np.median(neff_selected)
)

print(
    "p10 n_eff    =",
    np.quantile(neff_selected, 0.10)
)

print(
    "p90 n_eff    =",
    np.quantile(neff_selected, 0.90)
)

print(
    "min n_eff    =",
    np.min(neff_selected)
)

print(
    "max n_eff    =",
    np.max(neff_selected)
)

assert np.all(
    np.isfinite(neff_selected)
)

# ============================================================
# 3. Recompute selected prediction intervals
# ============================================================

q_selected = weighted_q_matrix(
    W_selected,
    tau=TAU_LOCAL
)

lower_selected = (
    val_pred_arr
    - q_selected
)

upper_selected = (
    val_pred_arr
    + q_selected
)

covered_selected = (
    (val_true_arr >= lower_selected)
    &
    (val_true_arr <= upper_selected)
)

width_selected = (
    upper_selected
    - lower_selected
)

score_selected = interval_score(
    val_true_arr,
    lower_selected,
    upper_selected,
    alpha=ALPHA
)

coverage_selected = np.mean(
    covered_selected
)

mean_width_selected = np.mean(
    width_selected
)

mean_score_selected = np.mean(
    score_selected
)

# ============================================================
# 4. Must exactly match STEP 10
# ============================================================

print("\n" + "=" * 80)
print("STEP 10 REPRODUCTION CHECK")
print("=" * 80)

check_df = pd.DataFrame([
    {
        "source": "STEP10_grid",
        "coverage":
            best_additive["coverage"],
        "mean_width":
            best_additive["mean_width"],
        "interval_score":
            best_additive["interval_score"],
    },
    {
        "source": "STEP10C_recomputed",
        "coverage":
            coverage_selected,
        "mean_width":
            mean_width_selected,
        "interval_score":
            mean_score_selected,
    }
])

display(check_df)

print(
    "ΔCoverage =",
    coverage_selected
    - best_additive["coverage"]
)

print(
    "ΔWidth =",
    mean_width_selected
    - best_additive["mean_width"]
)

print(
    "ΔScore =",
    mean_score_selected
    - best_additive["interval_score"]
)

assert np.isclose(
    coverage_selected,
    best_additive["coverage"],
    atol=1e-12
)

assert np.isclose(
    mean_width_selected,
    best_additive["mean_width"],
    atol=1e-12
)

assert np.isclose(
    mean_score_selected,
    best_additive["interval_score"],
    atol=1e-12
)

print(
    "\n✅ Selected Additive-EUC successfully reproduced"
)

# ============================================================
# 5. Correct canonical result
# ============================================================

additive_canonical_v3 = pd.DataFrame([{
    "method":
        "Additive-EUC",

    "h_geo_km":
        best_h_geo_add,

    "h_feat":
        best_h_feat_add,

    "lambda":
        best_lambda_add,

    "coverage":
        coverage_selected,

    "mean_width":
        mean_width_selected,

    "interval_score":
        mean_score_selected,

    "median_n_eff":
        np.median(neff_selected),

    "p10_n_eff":
        np.quantile(
            neff_selected,
            0.10
        ),

    "p90_n_eff":
        np.quantile(
            neff_selected,
            0.90
        ),

    "coverage_constraint_feasible":
        False,

    "selection_rule":
        "fallback_global_min_interval_score",

    "geo_distance":
        "Haversine_km",

    "feature_distance":
        "standardized_Euclidean",

    "fusion":
        "lambda*K_geo+(1-lambda)*K_feat",
}])

print("\n" + "=" * 80)
print("CORRECT CANONICAL ADDITIVE-EUC RESULT")
print("=" * 80)

display(
    additive_canonical_v3
)

additive_canonical_v3.to_csv(
    OUTPUT_DIR
    / "pm25_formal_additive_euc_validation_best_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 6. Explicitly keep lambda=0 equivalence as diagnostic only
# ============================================================

print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)

print(
    "STEP 10B lambda=0 equivalence remains valid "
    "ONLY as a mathematical sanity check."
)

print(
    "It does NOT describe the selected Additive-EUC model, "
    "because selected lambda =",
    best_lambda_add
)

print("\n✅ STEP 10C 完成")
print("✅ Additive-EUC 正式 Validation 结果已纠正")
print("⚠️ Frozen Test y 仍未读取")

STEP 10C — CORRECT ADDITIVE-EUC SELECTED RESULT
Selected parameters from STEP 10:
h_geo  = 150.9941416740361
h_feat = 1.6382459437843868
lambda = 0.2

✅ Selected Additive-EUC is a genuine geo + feature fusion solution

SELECTED ADDITIVE n_eff
median n_eff = 83.25320450289948
p10 n_eff    = 43.54875564605717
p90 n_eff    = 96.26166545152356
min n_eff    = 1.6607371130565385
max n_eff    = 103.90667879699146

STEP 10 REPRODUCTION CHECK


,source,coverage,mean_width,interval_score
0,STEP10_grid,0.823944,17.027888,31.899938
1,STEP10C_recomputed,0.823944,17.027888,31.899938


ΔCoverage = 0.0
ΔWidth = 0.0
ΔScore = 0.0

✅ Selected Additive-EUC successfully reproduced

CORRECT CANONICAL ADDITIVE-EUC RESULT


,method,h_geo_km,h_feat,lambda,coverage,mean_width,interval_score,median_n_eff,p10_n_eff,p90_n_eff,coverage_constraint_feasible,selection_rule,geo_distance,feature_distance,fusion
0,Additive-EUC,150.994142,1.638246,0.2,0.823944,17.027888,31.899938,83.253205,43.548756,96.261665,False,fallback_global_min_interval_score,Haversine_km,standardized_Euclidean,lambda*K_geo+(1-lambda)*K_feat



INTERPRETATION
STEP 10B lambda=0 equivalence remains valid ONLY as a mathematical sanity check.
It does NOT describe the selected Additive-EUC model, because selected lambda = 0.2

✅ STEP 10C 完成
✅ Additive-EUC 正式 Validation 结果已纠正
⚠️ Frozen Test y 仍未读取


In [13]:
# ============================================================
# STEP 11 — Joint-HAV-EUC architecture control
#
# Same information as Additive-EUC:
#   geographic = Haversine km
#   feature    = standardized Euclidean
#
# Difference:
#   Joint-HAV-EUC:
#       d_joint = sqrt(lambda*d_geo^2
#                    + (1-lambda)*d_feat^2)
#       one shared bandwidth h
#
# Validation only
# Frozen Test untouched
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 11 — JOINT-HAV-EUC ARCHITECTURE CONTROL")
print("=" * 80)

ALPHA = 0.1
TARGET_COVERAGE = 0.90

# ------------------------------------------------------------
# 1. Same lambda grid as Additive-EUC
# ------------------------------------------------------------

JOINT_LAMBDA_GRID = np.asarray(
    LAMBDA_ADDITIVE_GRID,
    dtype=float
)

print("Lambda grid:")
print(JOINT_LAMBDA_GRID)

# ------------------------------------------------------------
# 2. Inputs
# ------------------------------------------------------------

assert D_geo_val_calib.shape == D_feat_euc.shape

residuals = np.asarray(
    calib_residual,
    dtype=float
)

val_pred_arr = np.asarray(
    pred_val,
    dtype=float
)

val_true_arr = np.asarray(
    y_val,
    dtype=float
)

# ------------------------------------------------------------
# 3. For each lambda, construct bandwidth candidates
#
# Bandwidth grid is determined ONLY by the corresponding
# joint-distance matrix. Validation y is not used.
#
# This avoids forcing the same numerical h scale onto
# very different lambda values.
# ------------------------------------------------------------

rows = []
joint_payload = {}

bandwidth_grid_records = []

for lam in JOINT_LAMBDA_GRID:

    D_joint = np.sqrt(
        lam * D_geo_val_calib ** 2
        +
        (1.0 - lam) * D_feat_euc ** 2
    )

    nearest_joint = D_joint.min(
        axis=1
    )

    all_joint = D_joint.ravel()

    h_min_joint = max(
        0.05,
        0.5 * np.quantile(
            nearest_joint,
            0.05
        )
    )

    h_max_joint = np.quantile(
        all_joint,
        0.90
    )

    # 防止极端情况下上下限相同
    if np.isclose(
        h_min_joint,
        h_max_joint
    ):
        h_max_joint = (
            h_min_joint * 1.01
        )

    H_JOINT_GRID = np.geomspace(
        h_min_joint,
        h_max_joint,
        20
    )

    for h in H_JOINT_GRID:

        bandwidth_grid_records.append({
            "lambda": float(lam),
            "bandwidth": float(h),
        })

        # Stable Gaussian:
        # joint kernel only has one component,
        # so common max-shift is valid.
        log_w = (
            -0.5
            * (D_joint / h) ** 2
        )

        log_w = (
            log_w
            - np.max(
                log_w,
                axis=1,
                keepdims=True
            )
        )

        W = np.exp(
            log_w
        )

        # ----------------------------------------------------
        # weighted q
        # ----------------------------------------------------

        q_local = weighted_q_matrix(
            W,
            tau=TAU_LOCAL
        )

        # Stable effective sample size
        neff = neff_matrix_stable(
            W
        )

        lower_v = (
            val_pred_arr
            - q_local
        )

        upper_v = (
            val_pred_arr
            + q_local
        )

        covered_v = (
            (val_true_arr >= lower_v)
            &
            (val_true_arr <= upper_v)
        )

        width_v = (
            upper_v
            - lower_v
        )

        iscore_v = interval_score(
            val_true_arr,
            lower_v,
            upper_v,
            alpha=ALPHA
        )

        row = {
            "lambda":
                float(lam),

            "bandwidth":
                float(h),

            "coverage":
                np.mean(
                    covered_v
                ),

            "mean_half_width":
                np.mean(
                    q_local
                ),

            "mean_width":
                np.mean(
                    width_v
                ),

            "interval_score":
                np.mean(
                    iscore_v
                ),

            "median_n_eff":
                np.median(
                    neff
                ),

            "p10_n_eff":
                np.quantile(
                    neff,
                    0.10
                ),

            "p90_n_eff":
                np.quantile(
                    neff,
                    0.90
                ),
        }

        rows.append(row)

        joint_payload[
            (
                float(lam),
                float(h)
            )
        ] = {
            "q_local": q_local,
            "lower": lower_v,
            "upper": upper_v,
            "covered": covered_v,
            "width": width_v,
            "interval_score": iscore_v,
            "n_eff": neff,
        }

joint_hav_grid = pd.DataFrame(
    rows
)

print(
    "\nTotal combinations:",
    len(joint_hav_grid)
)

# ------------------------------------------------------------
# 4. Same frozen selection rule
# ------------------------------------------------------------

feasible = joint_hav_grid[
    joint_hav_grid["coverage"]
    >= TARGET_COVERAGE
]

if len(feasible) > 0:

    best_idx = (
        feasible[
            "interval_score"
        ].idxmin()
    )

    joint_feasible = True

    joint_selection_rule = (
        "coverage>=0.90_then_min_interval_score"
    )

else:

    best_idx = (
        joint_hav_grid[
            "interval_score"
        ].idxmin()
    )

    joint_feasible = False

    joint_selection_rule = (
        "fallback_global_min_interval_score"
    )

best_joint = (
    joint_hav_grid
    .loc[best_idx]
)

best_joint_lambda = float(
    best_joint["lambda"]
)

best_joint_h = float(
    best_joint["bandwidth"]
)

# ------------------------------------------------------------
# 5. Output
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("JOINT-HAV-EUC COVERAGE FEASIBILITY")
print("=" * 80)

print(
    "Coverage >= 0.90 feasible:",
    joint_feasible
)

print(
    "Maximum validation coverage:",
    joint_hav_grid[
        "coverage"
    ].max()
)

print(
    "Selection rule:",
    joint_selection_rule
)

print("\n" + "=" * 80)
print("FORMAL JOINT-HAV-EUC — SELECTED PARAMETER")
print("=" * 80)

print(
    f"Lambda         = "
    f"{best_joint_lambda:.6f}"
)

print(
    f"Bandwidth      = "
    f"{best_joint_h:.6f}"
)

print(
    f"Coverage       = "
    f"{best_joint['coverage']:.6f}"
)

print(
    f"Mean Width     = "
    f"{best_joint['mean_width']:.6f}"
)

print(
    f"Interval Score = "
    f"{best_joint['interval_score']:.6f}"
)

print(
    f"Median n_eff   = "
    f"{best_joint['median_n_eff']:.6f}"
)

# ------------------------------------------------------------
# 6. Boundary diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("JOINT BOUNDARY DIAGNOSTIC")
print("=" * 80)

print(
    "lambda boundary:",
    (
        np.isclose(
            best_joint_lambda,
            0.0
        )
        or
        np.isclose(
            best_joint_lambda,
            1.0
        )
    )
)

if np.isclose(
    best_joint_lambda,
    0.0
):
    print(
        "⚠️ selected Joint-HAV-EUC is feature-only"
    )

elif np.isclose(
    best_joint_lambda,
    1.0
):
    print(
        "⚠️ selected Joint-HAV-EUC is geography-only"
    )

# ------------------------------------------------------------
# 7. Strict architecture comparison
# ------------------------------------------------------------

architecture_comparison = pd.DataFrame([
    {
        "method":
            "Joint-HAV-EUC",

        "coverage":
            best_joint[
                "coverage"
            ],

        "mean_width":
            best_joint[
                "mean_width"
            ],

        "interval_score":
            best_joint[
                "interval_score"
            ],

        "median_n_eff":
            best_joint[
                "median_n_eff"
            ],
    },

    {
        "method":
            "Additive-EUC",

        "coverage":
            coverage_selected,

        "mean_width":
            mean_width_selected,

        "interval_score":
            mean_score_selected,

        "median_n_eff":
            np.median(
                neff_selected
            ),
    },
])

print("\n" + "=" * 80)
print("STRICT ARCHITECTURE COMPARISON")
print("=" * 80)

display(
    architecture_comparison
)

# ------------------------------------------------------------
# 8. Relative differences
# ------------------------------------------------------------

joint_width = float(
    best_joint[
        "mean_width"
    ]
)

joint_score = float(
    best_joint[
        "interval_score"
    ]
)

joint_cov = float(
    best_joint[
        "coverage"
    ]
)

print("\nAdditive relative to Joint-HAV-EUC:")

print(
    "Coverage change (pp) =",
    100 * (
        coverage_selected
        - joint_cov
    )
)

print(
    "Width change (%)     =",
    100 * (
        mean_width_selected
        / joint_width
        - 1
    )
)

print(
    "Score change (%)     =",
    100 * (
        mean_score_selected
        / joint_score
        - 1
    )
)

# ------------------------------------------------------------
# 9. Save
# ------------------------------------------------------------

joint_hav_grid.to_csv(
    OUTPUT_DIR
    / "pm25_formal_joint_hav_euc_validation_grid_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(
    bandwidth_grid_records
).to_csv(
    OUTPUT_DIR
    / "pm25_formal_joint_hav_bandwidth_grids_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    **best_joint.to_dict(),

    "coverage_constraint_feasible":
        joint_feasible,

    "selection_rule":
        joint_selection_rule,

    "geo_distance":
        "Haversine_km",

    "feature_distance":
        "standardized_Euclidean",

    "fusion":
        "joint_distance_shared_bandwidth",
}]).to_csv(
    OUTPUT_DIR
    / "pm25_formal_joint_hav_euc_validation_best_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

architecture_comparison.to_csv(
    OUTPUT_DIR
    / "pm25_formal_architecture_comparison_validation_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 11 完成")
print("✅ Joint vs Additive strict control 已建立")
print("⚠️ Frozen Test y 仍未读取")

STEP 11 — JOINT-HAV-EUC ARCHITECTURE CONTROL
Lambda grid:
[0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]

Total combinations: 220

JOINT-HAV-EUC COVERAGE FEASIBILITY
Coverage >= 0.90 feasible: False
Maximum validation coverage: 0.8591549295774648
Selection rule: fallback_global_min_interval_score

FORMAL JOINT-HAV-EUC — SELECTED PARAMETER
Lambda         = 0.000000
Bandwidth      = 1.638246
Coverage       = 0.823944
Mean Width     = 17.173541
Interval Score = 32.016616
Median n_eff   = 84.674013

JOINT BOUNDARY DIAGNOSTIC
lambda boundary: True
⚠️ selected Joint-HAV-EUC is feature-only

STRICT ARCHITECTURE COMPARISON


,method,coverage,mean_width,interval_score,median_n_eff
0,Joint-HAV-EUC,0.823944,17.173541,32.016616,84.674013
1,Additive-EUC,0.823944,17.027888,31.899938,83.253205



Additive relative to Joint-HAV-EUC:
Coverage change (pp) = 0.0
Width change (%)     = -0.8481200109922926
Score change (%)     = -0.3644287577922567

✅ STEP 11 完成
✅ Joint vs Additive strict control 已建立
⚠️ Frozen Test y 仍未读取
